# Evaluation Notebook

## 1. Setup & Environment

### 1.1 Mount Drive

Mount Google Drive for data and checkpoint access.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

### 1.2 Import Libraries

Import libraries grouped by purpose: file/data handling, numerical/tabular, PyTorch core, and experiment tracking.

In [ ]:
# File handling
import os
import sys
import json
import random
import shutil
import zipfile
from pathlib import Path

# Numerical / tabular
import numpy as np
import pandas as pd

# PyTorch core
import torch

# Experiment tracking
import wandb
import logging

# Metrics
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix, roc_curve,
)
from scipy import stats as scipy_stats

# Combinatorics / Math
import itertools
import math

# Visualisation
import matplotlib.pyplot as plt

In [ ]:
# Version check
print(f'torch: {torch.__version__}')
print(f'wandb: {wandb.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
  print(f'GPU: {torch.cuda.get_device_name(0)}')

### 1.3 Path Constants

All paths defined relative to the root, matching the previous notebooks' convention. `SRC` is added to `sys.path` so all imports are available.

In [ ]:
ROOT = Path('/content/drive/MyDrive/University/Masters/MSc_Thesis')
RESULTS = ROOT / 'outputs' / 'results'
SRC = ROOT / 'src'

sys.path.append(str(SRC))

print(f'ROOT: {ROOT}')
print(f'RESULTS: {RESULTS}')
print(f'SRC on path: {str(SRC) in sys.path}')

### 1.4 Reproducibility

Same global seed and `seed_everything()` function as in `03_training.ipynb` imported from `src/utils.py`. Used for the bootstrap resampling for statistical significance testing.

In [ ]:
from utils import seed_everything, SEED

seed_everything()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Global seed set: {SEED}')
print(f'Using device: {device}')

### 1.5 Model Building Code

Imported from `src/models.py` rather than redefined here.

In [ ]:
from models import build_model, ARCHITECTURE_BUILDERS, FREEZE_FUNCTIONS, get_output_linear

print('Imported from src/models.py:')
print(f'  Architectures: {list(ARCHITECTURE_BUILDERS.keys())}')
print(f'  Freeze functions: {list(FREEZE_FUNCTIONS.keys())}')

### 1.6 Experiment Tracking: Weights & Biases Setup

Authenticate via Colab Secrets. Identical code to `03_training.ipynb`.

In [ ]:
from google.colab import userdata
from utils import build_run_identity

WANDB_API_KEY = userdata.get('WANDB_API_KEY')
wandb.login(key=WANDB_API_KEY)
os.environ['WANDB_SILENT'] = 'true'

WANDB_ENTITY = 'thesis-mine'
WANDB_PROJECT = 'breast-ultrasound-classification'

### 1.7 Load Notebook 03 Outputs

Loads both `notebook03_results_summary.csv` and `final_models.json`, then cross-checks that they agree on every checkpoint path. They are redundant by design, but provide the possibility of a drift check.

In [ ]:
results_summary = pd.read_csv(RESULTS / 'notebook03_results_summary.csv')
final_models = json.load(open(RESULTS / 'final_models.json'))

print(f'results_summary: {results_summary.shape} (expected: (18, 22))')
print(f'final_models: {len(final_models)} entries (expected: 18)')

mismatches = []
for _, row in results_summary.iterrows():
  key = f"{row['architecture']}_{row['mode']}_{row['arm']}"
  fm_path = final_models.get(key, {}).get('checkpoint_path')
  if fm_path != row['final_checkpoint_path']:
    mismatches.append((key, fm_path, row['final_checkpoint_path']))

if mismatches:
  print(f'MISMATCH: {len(mismatches)} checkpoint path(s) disagree:')
  for key, fm_path, cs_path in mismatches:
    print(f'  {key}: final_models.json={fm_path}  vs  results_summary.csv={cs_path}')
  raise ValueError('final_models.json and notebook03_results_summary.csv disagree -- investigate before proceeding.')
else:
  print('Verified: final_models.json and notebook03_results_summary.csv agree on every checkpoint path.')

In [ ]:
# Sanity check: verify all 18 checkpoints load cleanly.

missing_checkpoints = []
load_errors = []

for _, row in results_summary.iterrows():
  checkpoint_path = Path(row['final_checkpoint_path'])
  # NaN, not None, is what a 'scratch' row's empty unfreeze_category cell
  # becomes after a CSV round-trip: build_model checks `is not None`,
  # so this has to be converted explicitly or scratch rows would wrongly
  # raise "mode='scratch' does not use unfreeze_category".
  unfreeze_category = row['unfreeze_category'] if pd.notna(row['unfreeze_category']) else None

  if not checkpoint_path.exists():
    missing_checkpoints.append(str(checkpoint_path))
    continue

  try:
    model = build_model(row['architecture'], row['mode'], row['dropout'],
                         unfreeze_category=unfreeze_category).to(device)
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    del model, state_dict
  except Exception as e:
    load_errors.append((row['architecture'], row['mode'], row['arm'], str(e)))

if missing_checkpoints or load_errors:
  if missing_checkpoints:
    print(f'Missing checkpoint files: {len(missing_checkpoints)}')
    for p in missing_checkpoints:
      print(f'  {p}')
  if load_errors:
    print(f'Load/shape errors: {len(load_errors)}')
    for arch, mode, arm, err in load_errors:
      print(f'  {arch}/{mode}/{arm}: {err}')
  raise ValueError('Not all 18 checkpoints loaded cleanly -- investigate before proceeding with evaluation.')
else:
  print(f'Verified: all {len(results_summary)} checkpoints load cleanly into their recorded architecture.')

## 2. Load & Verify Evaluation Data

### 2.1 Data Staging

Same zip-copy-and-extract pattern as `03_training.ipynb`, for faster read across the full evaluation sweep.

All 5 folders are covered in one step: all three BUSI arms, BUS-UCLM and BrEaST, all under the same `final.zip`.

In [ ]:
DRIVE_FINAL_DIR = ROOT / 'data' / 'final'
DRIVE_ZIP_PATH = DRIVE_FINAL_DIR.parent / 'final.zip'

LOCAL_DATA_DIR = Path('/content/data')
LOCAL_ZIP_PATH = Path('/content/final.zip')

print(f'Drive source: {DRIVE_FINAL_DIR}')
print(f'Drive source exists: {DRIVE_FINAL_DIR.exists()}')

In [ ]:
if DRIVE_ZIP_PATH.exists():
  print(f'Zip already exists on Drive: {DRIVE_ZIP_PATH}')
  print(f'Size: {DRIVE_ZIP_PATH.stat().st_size / (1024**3):.2f} GB')
else:
  print('No existing zip found -- creating one now.')
  archive_base = str(DRIVE_ZIP_PATH.with_suffix(''))
  shutil.make_archive(archive_base, 'zip', root_dir=DRIVE_FINAL_DIR)
  print(f'Zip created: {DRIVE_ZIP_PATH}')
  print(f'Size: {DRIVE_ZIP_PATH.stat().st_size / (1024**3):.2f} GB')

In [ ]:
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
print('Copied to local storage.')

with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zf:
  zf.extractall(LOCAL_DATA_DIR)
print('Extracted successfully.')

print(f'Local data directory: {LOCAL_DATA_DIR}')

### 2.2 Data Loading

`BreastUltrasoundDataset`, `build_transform`, `compute_dataset_stats`, `build_dataloader`/`seed_worker`, and the staged-data verification helpers (`EXPECTED_COUNTS`, `scan_local_counts`, `scan_class_folders`) are imported from `src/dataset_utils.py` rather than defined here.

In [ ]:
from dataset_utils import (
    CLASS_TO_IDX, EXPECTED_COUNTS, BreastUltrasoundDataset,
    scan_class_folders, scan_local_counts, build_transform,
    compute_dataset_stats, build_dataloader, IMAGENET_MEAN, IMAGENET_STD,
    BATCH_SIZE,
)

print('Imported from src/dataset_utils.py:')
print(f'  CLASS_TO_IDX: {CLASS_TO_IDX}')
print(f'  EXPECTED_COUNTS entries: {len(EXPECTED_COUNTS)}')

### 2.3 Verify Staged Data

Counts images per (dataset/arm, class) folder in the extracted local copy and checks against Notebook 02's confirmed final counts; re-verified here for safety reasons. Stops the notebook if anything doesn't match, rather than proceeding on unverified data.

In [ ]:
actual_counts = scan_local_counts(LOCAL_DATA_DIR)

In [ ]:
# Mismatch verification
mismatches = []

for key, expected in EXPECTED_COUNTS.items():
  actual = actual_counts[key]
  status = 'OK' if actual == expected else 'MISMATCH'
  if status == 'MISMATCH':
    mismatches.append((key, expected, actual))
  print(f'{key}: expected {expected}, found {actual} -- {status}')

total_expected = sum(EXPECTED_COUNTS.values())
total_actual = sum(v for v in actual_counts.values() if v is not None)
print(f'\nTotal expected: {total_expected}')
print(f'Total found: {total_actual}')

if mismatches:
  print(f'\n{len(mismatches)} mismatch(es) found:')
  for key, expected, actual in mismatches:
    print(f'  {key}: expected {expected}, got {actual}')
  raise ValueError('Staged data does not match Notebook 02 confirmed counts.')
else:
  print('\nVerified: all staged data matches Notebook 02 confirmed counts.')

### 2.4 Load BUSI Frozen Test Set + CV Pool

Loads the test/CV-pool split Notebook 03 computed once and froze to disk. There's no fallback branch that recomputes the split if the file is missing. If either file isn't there, that means something is genuinely wrong upstream.

`busi_cv_85_pool` is loaded here only for scratch-mode normalisation recomputation, considering it wasn't saved to the disk in Notebook 03. It serves no role in evaluation itself.

In [ ]:
test_path = RESULTS / 'busi_test_15.csv'
cv_pool_path = RESULTS / 'busi_cv_85_pool.csv'

if not test_path.exists():
  raise FileNotFoundError(f'Frozen test set not found at {test_path} -- this file must already exist from Notebook 03.')
if not cv_pool_path.exists():
  raise FileNotFoundError(f'CV pool not found at {cv_pool_path} -- this file must already exist from Notebook 03.')

busi_test = pd.read_csv(test_path)
busi_cv_pool = pd.read_csv(cv_pool_path)

print(f'busi_test: {len(busi_test)} images -- {busi_test["class"].value_counts().to_dict()}')
print(f'busi_cv_pool: {len(busi_cv_pool)} images -- {busi_cv_pool["class"].value_counts().to_dict()}')

overlap = set(busi_test['image_name']) & set(busi_cv_pool['image_name'])
if overlap:
  raise ValueError(f'Test/CV-pool overlap detected: {overlap}. This should be impossible for a frozen split -- investigate before proceeding.')
print('Verified: no overlap between test set and CV pool.')

### 2.5 Re-verify BUSI Arm Filename Consistency

Confirms all three BUSI arms still contain the exact same filenames per class. Verified here again, as to not rely on Notebook 03's output.

In [ ]:
BUSI_ARMS = ['BUSI_annotated', 'BUSI_inpainted', 'BUSI_masked']
CLASSES = ['benign', 'malignant', 'normal']

arm_dfs = {arm: scan_class_folders(LOCAL_DATA_DIR / 'BUSI' / arm, CLASSES) for arm in BUSI_ARMS}

for arm, df in arm_dfs.items():
  print(f'{arm}: {len(df)} images -- {df["class"].value_counts().to_dict()}')

In [ ]:
REFERENCE_ARM = 'BUSI_inpainted'

filename_sets = {arm: set(df['image_name']) for arm, df in arm_dfs.items()}
reference_set = filename_sets[REFERENCE_ARM]

set_mismatch = False
for arm, fset in filename_sets.items():
  if arm == REFERENCE_ARM:
    continue
  only_in_ref = reference_set - fset
  only_in_arm = fset - reference_set
  if only_in_ref or only_in_arm:
    set_mismatch = True
    print(f'MISMATCH -- {arm}:')
    if only_in_ref:
      print(f'  In {REFERENCE_ARM} but not {arm}: {only_in_ref}')
    if only_in_arm:
      print(f'  In {arm} but not {REFERENCE_ARM}: {only_in_arm}')

if set_mismatch:
  raise ValueError('BUSI arms no longer share identical filenames -- investigate before proceeding.')
else:
  print('Verified: all three BUSI arms share identical filenames per class.')

### 2.6 Build BUS-UCLM & BrEaST Dataframes

Built here for the first time by scanning each dataset's class folders directly, same approach as `scan_busi_arm`, generalised to a directory with no arm-level subfolder in between.

In [ ]:
bus_uclm_df = scan_class_folders(LOCAL_DATA_DIR / 'BUS-UCLM', CLASSES)
breast_df = scan_class_folders(LOCAL_DATA_DIR / 'BrEaST', CLASSES)

print(f'BUS-UCLM: {len(bus_uclm_df)} images -- {bus_uclm_df["class"].value_counts().to_dict()}')
print(f'BrEaST: {len(breast_df)} images -- {breast_df["class"].value_counts().to_dict()}')

In [ ]:
expected_bus_uclm = {cls: EXPECTED_COUNTS[('BUS-UCLM', cls)] for cls in CLASSES}
expected_breast = {cls: EXPECTED_COUNTS[('BrEaST', cls)] for cls in CLASSES}

actual_bus_uclm = bus_uclm_df['class'].value_counts().to_dict()
actual_breast = breast_df['class'].value_counts().to_dict()

if actual_bus_uclm != expected_bus_uclm:
  raise ValueError(f'BUS-UCLM class counts mismatch: expected {expected_bus_uclm}, got {actual_bus_uclm}')
if actual_breast != expected_breast:
  raise ValueError(f'BrEaST class counts mismatch: expected {expected_breast}, got {actual_breast}')

print('Verified: BUS-UCLM and BrEaST dataframes match expected counts.')

### 2.7 Resolve Per-Model Normalisation Stats

TL models normalise with fixed ImageNet mean/std. Scratch models need stats recomputed identically to how the final retrain computed them: once per arm, over the full, unfiltered `busi_cv_pool`. This computation is fully deterministic. Cached to Drive for convenience and to persist these values, since they were never saved during the original training run.

In [ ]:
ARMS = ['annotated', 'inpainted', 'masked']
scratch_stats_path = RESULTS / 'busi_scratch_normalization_stats.json'

if scratch_stats_path.exists():
  scratch_stats = json.load(open(scratch_stats_path))
  print('Loaded existing scratch-mode normalization stats from Drive -- not recomputed.')
else:
  scratch_stats = {}
  for arm in ARMS:
    base_dir = LOCAL_DATA_DIR / 'BUSI' / f'BUSI_{arm}'
    mean, std = compute_dataset_stats(busi_cv_pool, base_dir)
    scratch_stats[arm] = {'mean': mean, 'std': std}

  json.dump(scratch_stats, open(scratch_stats_path, 'w'), indent=2)
  print(f'Computed and saved scratch-mode normalization stats to {scratch_stats_path}')

for arm, stats in scratch_stats.items():
  print(f'{arm}: mean={stats["mean"]}, std={stats["std"]}')

In [ ]:
# Verification
for arm, stats in scratch_stats.items():
  mean, std = stats['mean'], stats['std']
  if not all(0 <= m <= 1 for m in mean) or not all(0 <= s <= 1 for s in std):
    raise ValueError(
        f'{arm}: implausible stats (mean={mean}, std={std}) -- expected values '
        f'in [0, 1] since ToTensor() scales pixel intensities to that range.'
    )
print('Verified: all scratch-mode stats fall within the plausible [0, 1] range.')

### 2.8 Eval Dataset/DataLoader Registry Build

Expands the 18 trained models into 54 evaluation jobs (18 models × 3 eval datasets: `busi_test`, `bus_uclm`, `breast`). Normalisation is resolved per model.

In [ ]:
EVAL_DATASETS = ['busi_test', 'bus_uclm', 'breast']

registry_rows = []

for _, model_row in results_summary.iterrows():
  architecture = model_row['architecture']
  mode = model_row['mode']
  arm = model_row['arm']

  # Normalisation is a property of the MODEL, never of the eval dataset:
  # every eval_data value below uses this same mean/std, including
  # bus_uclm/breast, which have no arm of their own.
  if mode == 'TL':
    mean, std = IMAGENET_MEAN, IMAGENET_STD
  else:
    mean, std = scratch_stats[arm]['mean'], scratch_stats[arm]['std']

  for eval_data in EVAL_DATASETS:
    if eval_data == 'busi_test':
      eval_df = busi_test
      base_dir = LOCAL_DATA_DIR / 'BUSI' / f'BUSI_{arm}'
    elif eval_data == 'bus_uclm':
      eval_df = bus_uclm_df
      base_dir = LOCAL_DATA_DIR / 'BUS-UCLM'
    elif eval_data == 'breast':
      eval_df = breast_df
      base_dir = LOCAL_DATA_DIR / 'BrEaST'

    registry_rows.append({
        'architecture': architecture,
        'mode': mode,
        'arm': arm,
        'dropout': model_row['dropout'],
        'unfreeze_category': model_row['unfreeze_category'],
        'checkpoint_path': model_row['final_checkpoint_path'],
        'eval_data': eval_data,
        'eval_df': eval_df,
        'base_dir': base_dir,
        'mean': mean,
        'std': std,
    })

eval_registry = pd.DataFrame(registry_rows)

expected_rows = len(results_summary) * len(EVAL_DATASETS)
print(f'eval_registry: {len(eval_registry)} rows (expected: {len(results_summary)} models x {len(EVAL_DATASETS)} = {expected_rows})')

In [ ]:
if len(eval_registry) != expected_rows:
  raise ValueError(f'Registry has {len(eval_registry)} rows, expected {expected_rows}.')

duplicate_count = eval_registry.duplicated(subset=['architecture', 'mode', 'arm', 'eval_data']).sum()
if duplicate_count > 0:
  raise ValueError(f'{duplicate_count} duplicate (architecture, mode, arm, eval_data) combination(s) found.')

missing_dirs = sorted({str(d) for d in eval_registry['base_dir'].unique() if not d.exists()})
if missing_dirs:
  raise ValueError(f'Missing base directories: {missing_dirs}')

print('Verified: 54 rows, no duplicate combinations, all base directories exist.')

In [ ]:
# Visual check on the normalisation-pairing logic specifically: TL rows
# should all show identical mean/std regardless of arm; scratch rows
# should show three distinct pairs, matching 2.7's printed values.
check_df = eval_registry.drop_duplicates(subset=['mode', 'arm'])[['mode', 'arm', 'mean', 'std']]
print(check_df.to_string(index=False))

### 2.9 Document the Eval-Dataset Mapping

Every one of the 18 models is evaluated on three datasets: its own arm's BUSI test set, plus BUS-UCLM and BrEaST. Verified through the code below.

In [ ]:
for eval_data in ['bus_uclm', 'breast']:
  unique_dirs = eval_registry[eval_registry['eval_data'] == eval_data]['base_dir'].unique()
  if len(unique_dirs) != 1:
    raise ValueError(f'{eval_data}: expected a single shared base_dir across all 18 models, found {len(unique_dirs)}: {unique_dirs}')
  print(f'{eval_data}: confirmed single shared base_dir across all 18 models -- {unique_dirs[0]}')

busi_test_dir_count = eval_registry[eval_registry['eval_data'] == 'busi_test']['base_dir'].nunique()
if busi_test_dir_count != 3:
  raise ValueError(f'busi_test: expected exactly 3 distinct base_dirs (one per arm), found {busi_test_dir_count}')
print('busi_test: confirmed 3 distinct base_dirs, one per arm.')

In [ ]:
mapping_summary = eval_registry.pivot_table(
    index=['architecture', 'mode', 'arm'],
    columns='eval_data',
    values='base_dir',
    aggfunc=lambda x: Path(x.iloc[0]).name,
)
print(mapping_summary.to_string())

## 3. Core Evaluation Pipeline

### 3.1 Model-Loading Helper

Builds a model from its recorded config, loads its checkpoint, and sets it to eval mode. `eval()` is set here unconditionally, not left to the caller, in order to eliminate the probability of silently failed results.

In [ ]:
import time

def load_eval_model(architecture, mode, dropout, unfreeze_category, checkpoint_path):
  # eval_registry copies unfreeze_category straight from results_summary.csv,
  # where a scratch-mode row's empty cell round-trips as NaN, not None. Normalising
  # here, at the point of consumption, covers every caller rather than relying on eval_registry
  # having handled it upstream.
  unfreeze_category = unfreeze_category if pd.notna(unfreeze_category) else None

  model = build_model(architecture, mode, dropout, unfreeze_category=unfreeze_category).to(device)
  state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
  model.load_state_dict(state_dict)
  model.eval()
  return model

### 3.2 Evaluation function

Runs one model over one evaluation dataset. Stores probabilities, not raw logits (necessary for the evaluation metrics). Stores `image_name` alongside every prediction, for alignment safety-check later. Latency is isolated to just the forward pass via `torch.cuda.synchronize()`, so it reflects real GPU compute time rather than dispatch overhead. Uses `torch.inference_mode()` rather than `torch.no_grad()`, which is the more efficient choice, considering the gradients aren't needed in this notebook.

In [ ]:
def evaluate_model(model, eval_df, base_dir, mean, std):
  if model.training:
    raise ValueError('Model is in training mode; call model.eval() before evaluate_model().')

  dataset = BreastUltrasoundDataset(eval_df, base_dir, build_transform('eval', mean, std))
  loader = build_dataloader(dataset, train=False)

  all_true, all_pred, all_probs, batch_latencies_ms = [], [], [], []

  with torch.inference_mode():
    for images, labels in loader:
      images = images.to(device)

      if device.type == 'cuda':
        torch.cuda.synchronize()
      start = time.perf_counter()
      logits = model(images)
      if device.type == 'cuda':
        torch.cuda.synchronize()
      batch_latencies_ms.append((time.perf_counter() - start) * 1000)

      probs = torch.softmax(logits, dim=1)
      preds = torch.argmax(probs, dim=1)

      all_true.extend(labels.tolist())
      all_pred.extend(preds.cpu().tolist())
      all_probs.extend(probs.cpu().tolist())

  image_names = eval_df['image_name'].tolist()
  if len(image_names) != len(all_true):
    raise ValueError(f'Image count mismatch: {len(image_names)} names vs {len(all_true)} predictions.')

  return {
      'image_name': image_names,
      'true_label': all_true,
      'pred_label': all_pred,
      'probabilities': all_probs,
      'batch_latencies_ms': batch_latencies_ms,
  }

### 3.3 Sanity Check on a Small Subset

Runs the real evaluation function against one real model and a small, class-stratified sample of real data, checking invariants that should always hold.

In [ ]:
sample_row = eval_registry[(eval_registry['mode'] == 'scratch') & (eval_registry['eval_data'] == 'bus_uclm')].iloc[0]

sample_model = load_eval_model(
    sample_row['architecture'], sample_row['mode'], sample_row['dropout'],
    sample_row['unfreeze_category'], sample_row['checkpoint_path'],
)

sample_df = sample_row['eval_df'].groupby('class').sample(n=3, random_state=SEED).reset_index(drop=True)

print(f"Sanity-checking: {sample_row['architecture']}/{sample_row['mode']}/{sample_row['arm']} on {sample_row['eval_data']}")
print(f"Sample size: {len(sample_df)} images -- {sample_df['class'].value_counts().to_dict()}")

sample_result = evaluate_model(sample_model, sample_df, sample_row['base_dir'], sample_row['mean'], sample_row['std'])

In [ ]:
n_images = len(sample_df)
n_expected_batches = -(-n_images // BATCH_SIZE)

checks = []
checks.append(('Prediction count matches image count',
                len(sample_result['pred_label']) == n_images))
checks.append(('Latency entries match expected batch count',
                len(sample_result['batch_latencies_ms']) == n_expected_batches))

prob_sums = [sum(p) for p in sample_result['probabilities']]
checks.append(('All probability rows sum to ~1.0',
                all(abs(s - 1.0) < 1e-5 for s in prob_sums)))

argmax_preds = [int(np.argmax(p)) for p in sample_result['probabilities']]
checks.append(('argmax(probabilities) matches recorded pred_label',
                argmax_preds == sample_result['pred_label']))

manual_true = [CLASS_TO_IDX[c] for c in sample_df['class']]
checks.append(('true_label matches an independent re-derivation from eval_df',
                manual_true == sample_result['true_label']))

checks.append(('image_name list matches eval_df, same length and order',
                sample_result['image_name'] == sample_df['image_name'].tolist()))

all_passed = True
for description, passed in checks:
  status = 'PASS' if passed else 'FAIL'
  print(f'[{status}] {description}')
  if not passed:
    all_passed = False

if not all_passed:
  raise ValueError('One or more sanity checks failed; do not proceed to the full sweep.')
print('\nAll sanity checks passed.')

del sample_model

### 3.4 Full Validation Run

Runs all 18 models across their applicable evaluation datasets (54 combinations total). Models are loaded once per checkpoint only, not once per combination. Each combination gets its own WandB run, with accuracy, malignant recall, F2 score, mean batch latency tracked as live monitoring only. Raw predictions and per-batch latencies are saved to `RESULTS/eval/`.

In [ ]:
logging.getLogger('wandb').setLevel(logging.ERROR)

EVAL_DIR = RESULTS / 'eval'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

FORCE_RECOMPUTE = False # set True to force a fresh sweep even if results already exists on Drive

In [ ]:
predictions_path = EVAL_DIR / 'predictions.csv'
latency_path = EVAL_DIR / 'batch_latencies.csv'

if predictions_path.exists() and latency_path.exists() and not FORCE_RECOMPUTE:
  print(f'Found existing results at {EVAL_DIR} -- loading instead of re-running the sweep.')
  print('(Set FORCE_RECOMPUTE = True above to force a fresh sweep.)')
  predictions_df = pd.read_csv(predictions_path)
  latency_df = pd.read_csv(latency_path)
  print(f'Loaded predictions_df: {predictions_df.shape}, latency_df: {latency_df.shape}')

else:
  model_groups = eval_registry.groupby('checkpoint_path')
  all_predictions = []
  all_latencies = []

  for checkpoint_path, group in model_groups:
    first_row = group.iloc[0]
    consistency_cols = ['architecture', 'mode', 'arm', 'dropout', 'unfreeze_category']
    if not group[consistency_cols].nunique(dropna=False).eq(1).all():
      raise ValueError(f'checkpoint_path {checkpoint_path} maps to inconsistent model config across rows.')

    architecture, mode, arm = first_row['architecture'], first_row['mode'], first_row['arm']
    dropout, unfreeze_category = first_row['dropout'], first_row['unfreeze_category']

    sweep_row = results_summary[results_summary['final_checkpoint_path'] == checkpoint_path].iloc[0]
    sweep_avg_val_f2 = sweep_row['sweep_avg_val_f2']
    cv_avg_val_f2 = sweep_row['cv_avg_val_f2']

    print(f'Loading {architecture}/{mode}/{arm}...')
    model = load_eval_model(architecture, mode, dropout, unfreeze_category, checkpoint_path)

    for _, row in group.iterrows():
      eval_data = row['eval_data']
      identity = build_run_identity(architecture, mode, arm, job_type='eval', eval_data=eval_data)

      run = wandb.init(
          entity=WANDB_ENTITY, project=WANDB_PROJECT,
          group=identity['group'], job_type=identity['job_type'],
          tags=identity['tags'], name=identity['name'],
          config={
              'architecture': architecture, 'mode': mode, 'arm': arm, 'eval_data': eval_data,
              'dropout': dropout, 'unfreeze_category': unfreeze_category,
          },
          settings=wandb.Settings(silent=True),
          reinit=True,
      )

      result = evaluate_model(model, row['eval_df'], row['base_dir'], row['mean'], row['std'])

      accuracy = accuracy_score(result['true_label'], result['pred_label'])
      malignant_recall = recall_score(
          result['true_label'], result['pred_label'],
          labels=[CLASS_TO_IDX['malignant']], average=None,
      )[0]
      macro_f2 = fbeta_score(result['true_label'], result['pred_label'], beta=2, average='macro')
      mean_latency_ms = float(np.mean(result['batch_latencies_ms']))

      wandb.log({
          'accuracy': accuracy, 'malignant_recall': malignant_recall, 'macro_f2': macro_f2,
          'sweep_avg_val_f2': sweep_avg_val_f2, 'cv_avg_val_f2': cv_avg_val_f2,
          'mean_batch_latency_ms': mean_latency_ms, 'n_images': len(result['true_label']),
      })
      run.finish()

      print(f'  {eval_data}: accuracy={accuracy:.3f}, malignant_recall={malignant_recall:.3f}, '
            f'macro_f2={macro_f2:.3f}, mean_latency={mean_latency_ms:.1f}ms')

      for i in range(len(result['image_name'])):
        all_predictions.append({
            'architecture': architecture, 'mode': mode, 'arm': arm, 'eval_data': eval_data,
            'image_name': result['image_name'][i],
            'true_label': result['true_label'][i], 'pred_label': result['pred_label'][i],
            'prob_benign': result['probabilities'][i][0],
            'prob_malignant': result['probabilities'][i][1],
            'prob_normal': result['probabilities'][i][2],
        })

      all_latencies.append({
          'architecture': architecture, 'mode': mode, 'arm': arm, 'eval_data': eval_data,
          'batch_latencies_ms': result['batch_latencies_ms'],
      })

    del model
    if device.type == 'cuda':
      torch.cuda.empty_cache()

  predictions_df = pd.DataFrame(all_predictions)
  latency_df = pd.DataFrame(all_latencies)
  predictions_df.to_csv(predictions_path, index=False)
  latency_df.to_csv(latency_path, index=False)
  print('\nSweep complete and saved.')

In [ ]:
expected_predictions = sum(len(row['eval_df']) for _, row in eval_registry.iterrows())
if len(predictions_df) != expected_predictions:
  raise ValueError(f'predictions_df has {len(predictions_df)} rows, expected {expected_predictions}.')

duplicate_count = predictions_df.duplicated(subset=['architecture', 'mode', 'arm', 'eval_data', 'image_name']).sum()
if duplicate_count > 0:
  raise ValueError(f'{duplicate_count} duplicate prediction row(s) found.')

if len(latency_df) != len(eval_registry):
  raise ValueError(f'latency_df has {len(latency_df)} rows, expected {len(eval_registry)}.')

print(f'predictions.csv: {len(predictions_df)} rows -- verified, no duplicates.')
print(f'batch_latencies.csv: {len(latency_df)} rows -- one per (model, eval_data) combination.')
print(f'\nSaved to: {EVAL_DIR}')

## 4. Technical Metrics

### 4.1 Metric functions

One function per metric family: hard-prediction metrics, probability-based metrics, confusion matrices; each verified against a constructed case with hand-derived expected values before touching real data. The `y_true`/`y_pred` pair reuses verified macro-F2 test case from Notebook 03.

In [ ]:
CLASS_NAMES = ['benign', 'malignant', 'normal']

def compute_baseline_accuracy(y_true):
  """Accuracy of always predicting the majority class: the 'no-information rate'."""
  y_true = np.asarray(y_true)
  _, counts = np.unique(y_true, return_counts=True)
  return counts.max() / len(y_true)


def compute_argmax_metrics(y_true, y_pred):
  """Accuracy, baseline, macro-F1/F2, per-class precision/recall/F1."""
  metrics = {
      'accuracy': accuracy_score(y_true, y_pred),
      'baseline_accuracy': compute_baseline_accuracy(y_true),
      'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0, labels=[0, 1, 2]),
      'macro_f2': fbeta_score(y_true, y_pred, beta=2, average='macro', zero_division=0, labels=[0, 1, 2]),
  }
  precision = precision_score(y_true, y_pred, average=None, zero_division=0, labels=[0, 1, 2])
  recall = recall_score(y_true, y_pred, average=None, zero_division=0, labels=[0, 1, 2])
  f1 = f1_score(y_true, y_pred, average=None, zero_division=0, labels=[0, 1, 2])
  for i, cls in enumerate(CLASS_NAMES):
    metrics[f'precision_{cls}'] = precision[i]
    metrics[f'recall_{cls}'] = recall[i]
    metrics[f'f1_{cls}'] = f1[i]
  return metrics


def compute_probability_metrics(y_true, y_prob):
  """Per-class + macro AUC-ROC, per-class PR-AUC."""
  y_true = np.asarray(y_true)
  y_prob = np.asarray(y_prob)
  metrics = {}
  auc_per_class = []
  for i, cls in enumerate(CLASS_NAMES):
    y_true_binary = (y_true == i).astype(int)
    auc = roc_auc_score(y_true_binary, y_prob[:, i])
    ap = average_precision_score(y_true_binary, y_prob[:, i])
    metrics[f'auc_{cls}'] = auc
    metrics[f'pr_auc_{cls}'] = ap
    auc_per_class.append(auc)
  metrics['auc_macro'] = np.mean(auc_per_class)
  return metrics


def compute_confusion_matrices(y_true, y_pred):
  """Raw counts and row-normalised percentages, 3x3, order [benign, malignant, normal]."""
  raw = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
  row_sums = raw.sum(axis=1, keepdims=True)
  normalized = np.divide(raw, row_sums, out=np.zeros_like(raw, dtype=float), where=row_sums != 0)
  return raw, normalized


def macro_f2_n03(y_true, y_pred, num_classes=3, beta=2):
  """N03's exact hand-implemented macro-F2 (03_training.ipynb, 5.2), copied
  verbatim for a direct cross-check against sklearn's fbeta_score below."""
  y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
  f2_per_class = []
  for cls in range(num_classes):
    tp = np.sum((y_pred == cls) & (y_true == cls))
    fp = np.sum((y_pred == cls) & (y_true != cls))
    fn = np.sum((y_pred != cls) & (y_true == cls))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f2 = 0.0 if precision + recall == 0 else (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall)
    f2_per_class.append(f2)
  return np.mean(f2_per_class)

In [ ]:
# Verification
y_true_test = [0, 0, 1, 1, 2, 2]
y_pred_test = [0, 0, 0, 0, 2, 2]
y_prob_test = [
    [0.70, 0.20, 0.10], [0.60, 0.30, 0.10],
    [0.52, 0.28, 0.20], [0.45, 0.40, 0.15],
    [0.12, 0.18, 0.70], [0.15, 0.15, 0.70],
]

argmax_metrics = compute_argmax_metrics(y_true_test, y_pred_test)
prob_metrics = compute_probability_metrics(y_true_test, y_prob_test)
raw_cm, norm_cm = compute_confusion_matrices(y_true_test, y_pred_test)
all_metrics = {**argmax_metrics, **prob_metrics}

expected = {
    'accuracy': 0.6667, 'baseline_accuracy': 0.3333, 'macro_f1': 0.5556, 'macro_f2': 0.6111,
    'precision_benign': 0.5, 'recall_benign': 1.0, 'f1_benign': 0.6667,
    'precision_malignant': 0.0, 'recall_malignant': 0.0, 'f1_malignant': 0.0,
    'precision_normal': 1.0, 'recall_normal': 1.0, 'f1_normal': 1.0,
    'auc_benign': 1.0, 'auc_malignant': 0.875, 'auc_normal': 1.0, 'auc_macro': 0.9583,
    'pr_auc_benign': 1.0, 'pr_auc_malignant': 0.8333, 'pr_auc_normal': 1.0,
}

all_passed = True
for key, exp_val in expected.items():
  if abs(all_metrics[key] - exp_val) >= 1e-3:
    all_passed = False
    print(f'[FAIL] {key}: expected {exp_val}, got {all_metrics[key]:.4f}')

if not np.array_equal(raw_cm, np.array([[2, 0, 0], [2, 0, 0], [0, 0, 2]])):
  all_passed = False
  print(f'[FAIL] raw confusion matrix:\n{raw_cm}')
if not np.allclose(norm_cm, np.array([[1, 0, 0], [1, 0, 0], [0, 0, 1]])):
  all_passed = False
  print(f'[FAIL] normalized confusion matrix:\n{norm_cm}')

n03_f2 = macro_f2_n03(y_true_test, y_pred_test)
if abs(n03_f2 - all_metrics['macro_f2']) > 1e-9:
  all_passed = False
  print(f'[FAIL] N03 macro_f2 ({n03_f2}) vs sklearn ({all_metrics["macro_f2"]})')

if all_passed:
  print('All technical metric functions verified against hand-derived expected values.')
  print(f'N03 custom macro_f2 matches sklearn exactly: {n03_f2:.4f}')
else:
  raise ValueError('One or more checks failed. Do not proceed.')

### 4.2 Metrics Applied Across All 54 Combinations

Groups `predictions_df` by (architecture, mode, arm, eval_data), runs every technical metric function per group, collects into one long-format table.

In [ ]:
technical_results = []
grouped = predictions_df.groupby(['architecture', 'mode', 'arm', 'eval_data'])

for (architecture, mode, arm, eval_data), group in grouped:
  y_true = group['true_label'].values
  y_pred = group['pred_label'].values
  y_prob = group[['prob_benign', 'prob_malignant', 'prob_normal']].values

  row = {'architecture': architecture, 'mode': mode, 'arm': arm, 'eval_data': eval_data, 'n_images': len(group)}
  row.update(compute_argmax_metrics(y_true, y_pred))
  row.update(compute_probability_metrics(y_true, y_prob))

  raw_cm, _ = compute_confusion_matrices(y_true, y_pred)
  for i, true_cls in enumerate(CLASS_NAMES):
    for j, pred_cls in enumerate(CLASS_NAMES):
      row[f'cm_true_{true_cls}_pred_{pred_cls}'] = int(raw_cm[i, j])

  technical_results.append(row)

technical_df = pd.DataFrame(technical_results)
print(f'technical_df: {technical_df.shape}')

In [ ]:
if len(technical_df) != 54:
  raise ValueError(f'Expected 54 rows, got {len(technical_df)}.')

metric_cols = [c for c in technical_df.columns if c not in ['architecture', 'mode', 'arm', 'eval_data']]
if technical_df[metric_cols].isna().any().any():
  raise ValueError('NaN found in computed metrics.')

expected_n = eval_registry.set_index(['architecture', 'mode', 'arm', 'eval_data'])['eval_df'].apply(len)
merged_n = technical_df.set_index(['architecture', 'mode', 'arm', 'eval_data'])['n_images']
if not (merged_n == expected_n.reindex(merged_n.index)).all():
  raise ValueError('n_images mismatch against eval_registry.')

cm_cols = [c for c in technical_df.columns if c.startswith('cm_true_')]
if not (technical_df[cm_cols].sum(axis=1) == technical_df['n_images']).all():
  raise ValueError('Confusion matrix counts do not sum to n_images.')

spot = technical_df[(technical_df['architecture'] == 'resnet50') & (technical_df['mode'] == 'TL') &
                     (technical_df['arm'] == 'masked') & (technical_df['eval_data'] == 'busi_test')].iloc[0]
if abs(spot['accuracy'] - 0.875) > 1e-3 or abs(spot['macro_f2'] - 0.860) > 1e-3:
  raise ValueError(f"Spot-check mismatch vs 3.4 output: accuracy={spot['accuracy']:.3f}, macro_f2={spot['macro_f2']:.3f}")

print('All checks passed: 54 rows, no NaNs, n_images/CM sums consistent, spot-check matches 3.4.')

technical_df.to_csv(EVAL_DIR / 'technical_metrics.csv', index=False)
print(f'Saved to {EVAL_DIR / "technical_metrics.csv"}')

### 4.3 Confusion Matrix Export

One combined-format image per combination (count + row-percentage in each cell), saved under `RESULTS/eval/confusion_matrices/`. Filenames follow the same `{architecture}_{mode}_{arm}_{eval_data}` pattern already used for WandB run names, for consistency. Colour scale uses row-normalised values (not raw counts), so intensity is visually comparable across datasets of very different sizes.

In [ ]:
CM_DIR = EVAL_DIR / 'confusion_matrices'
CM_DIR.mkdir(parents=True, exist_ok=True)

def text_color_for_bg(rgba):
  r, g, b = rgba[:3]
  luminance = 0.299 * r + 0.587 * g + 0.114 * b
  return 'white' if luminance < 0.5 else 'black'


def plot_confusion_matrix(raw_cm, normalized_cm, title, save_path):
  cmap = plt.get_cmap('YlOrBr')
  fig, ax = plt.subplots(figsize=(5, 4.5))
  im = ax.imshow(normalized_cm, cmap=cmap, vmin=0, vmax=1)

  ax.set_xticks(range(3))
  ax.set_yticks(range(3))
  ax.set_xticklabels(CLASS_NAMES)
  ax.set_yticklabels(CLASS_NAMES)
  ax.set_xlabel('Predicted')
  ax.set_ylabel('True')
  ax.set_title(title, fontsize=10)

  for i in range(3):
    for j in range(3):
      count = raw_cm[i, j]
      pct = normalized_cm[i, j] * 100
      color = text_color_for_bg(cmap(normalized_cm[i, j]))
      ax.text(j, i, f'{count}\n({pct:.1f}%)', ha='center', va='center', color=color, fontsize=9)

  fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
  fig.tight_layout()
  fig.savefig(save_path, dpi=150)
  plt.close(fig)

for (architecture, mode, arm, eval_data), group in predictions_df.groupby(['architecture', 'mode', 'arm', 'eval_data']):
  y_true = group['true_label'].values
  y_pred = group['pred_label'].values
  raw_cm, normalized_cm = compute_confusion_matrices(y_true, y_pred)

  title = f'{architecture} / {mode} / {arm}\n{eval_data} (n={len(group)})'
  filename = f'{architecture}_{mode}_{arm}_{eval_data}.png'
  plot_confusion_matrix(raw_cm, normalized_cm, title, CM_DIR / filename)

print(f'Confusion matrices saved to {CM_DIR}')

In [ ]:
saved_files = list(CM_DIR.glob('*.png'))
if len(saved_files) != 54:
  raise ValueError(f'Expected 54 confusion matrix images, found {len(saved_files)}.')

sample_file = CM_DIR / 'resnet50_TL_masked_busi_test.png'
if not sample_file.exists():
  raise ValueError(f'Expected file not found: {sample_file}')
if sample_file.stat().st_size < 1000:
  raise ValueError(f'{sample_file} is suspiciously small ({sample_file.stat().st_size} bytes) -- likely blank/corrupt.')

print(f'Verified: 54 confusion matrix images saved, spot-checked file exists and is non-trivial size.')

#### Verification of Existence on Drive and in Memory

In [ ]:
# Round-trip: does the saved CSV match what's actually in memory?
reloaded_df = pd.read_csv(EVAL_DIR / 'technical_metrics.csv')
if reloaded_df.shape != technical_df.shape:
  raise ValueError(f'Shape mismatch: saved {reloaded_df.shape}, in-memory {technical_df.shape}')

numeric_cols = technical_df.select_dtypes(include=[np.number]).columns
if not np.allclose(reloaded_df[numeric_cols].values, technical_df[numeric_cols].values, atol=1e-9):
  raise ValueError('Saved CSV values differ from in-memory technical_df beyond floating-point tolerance.')

n_cm_files = len(list(CM_DIR.glob('*.png')))
if n_cm_files != 54:
  raise ValueError(f'Expected 54 confusion matrix files on disk, found {n_cm_files}.')

print('Round-trip verified: saved files match in-memory results.')

# Plausibility scan -- structure already verified; this checks the VALUES.
suspect = technical_df[(technical_df['auc_macro'] < 0.5) | (technical_df['accuracy'] < technical_df['baseline_accuracy'] - 0.05)]
if len(suspect) > 0:
  print(f'{len(suspect)} row(s) flagged (AUC below chance, or accuracy below baseline):')
  print(suspect[['architecture', 'mode', 'arm', 'eval_data', 'accuracy', 'baseline_accuracy', 'auc_macro']].to_string(index=False))
else:
  print('No rows flagged.')

print(f"\naccuracy  range: {technical_df['accuracy'].min():.3f} - {technical_df['accuracy'].max():.3f}")
print(f"macro_f2  range: {technical_df['macro_f2'].min():.3f} - {technical_df['macro_f2'].max():.3f}")
print(f"auc_macro range: {technical_df['auc_macro'].min():.3f} - {technical_df['auc_macro'].max():.3f}")

## 5. Clinical Metrics

### 5.1 Sensitivity at Fixed Specificity

Malignant-vs-rest sensitivity at 85% / 90% / 95% specificity, with the operating threshold recorded alongside each value. Filters out sklearn's `threshold=inf` point before interpolating, since the specificity targets sit near exactly that end of the curve.

In [ ]:
MALIGNANT_IDX = CLASS_TO_IDX['malignant']
TARGET_SPECIFICITIES = [0.85, 0.90, 0.95]

def compute_sensitivity_at_specificity(y_true_binary, y_score, targets=TARGET_SPECIFICITIES):
  fpr, tpr, thresholds = roc_curve(y_true_binary, y_score, drop_intermediate=False)
  finite = np.isfinite(thresholds)
  fpr, tpr, thresholds = fpr[finite], tpr[finite], thresholds[finite]
  specificity = 1 - fpr

  # Vertical segments (tied specificity, differing sensitivity) break
  # np.interp's monotonicity assumption. Collapse to one row per unique
  # specificity -- max tpr (best achievable), paired with min threshold
  # (the one that actually achieves it; within a tie group, lower
  # threshold always corresponds to higher tpr).
  # Numpy-only version; ~8x faster than the orignal groupby version.
  unique_spec, inverse = np.unique(specificity, return_inverse=True)
  agg_tpr = np.full(len(unique_spec), -np.inf)
  agg_thresh = np.full(len(unique_spec), np.inf)
  np.maximum.at(agg_tpr, inverse, tpr)
  np.minimum.at(agg_thresh, inverse, thresholds)

  if not np.all(np.diff(agg_tpr) <= 1e-9):
    raise ValueError('tpr not monotonically non-increasing after de-duplication -- do not trust interpolation.')

  results = {}
  for target in targets:
    results[target] = {
        'sensitivity': float(np.interp(target, unique_spec, agg_tpr)),
        'threshold': float(np.interp(target, unique_spec, agg_thresh)),
    }
  return results

### 5.2 False Negative Rate, PPV/NPV

FNR and PPV/NPV read directly from the confusion matrix cells already in `technical_df`. PPV/NPV reported alongside `malignant_prevalence` on every row.

In [ ]:
clinical_results = []

for (architecture, mode, arm, eval_data), group in predictions_df.groupby(['architecture', 'mode', 'arm', 'eval_data']):
  y_true_binary = (group['true_label'].values == MALIGNANT_IDX).astype(int)
  y_score = group['prob_malignant'].values

  sens_spec = compute_sensitivity_at_specificity(y_true_binary, y_score)

  tech_row = technical_df[(technical_df['architecture'] == architecture) & (technical_df['mode'] == mode) &
                           (technical_df['arm'] == arm) & (technical_df['eval_data'] == eval_data)].iloc[0]

  tp = tech_row['cm_true_malignant_pred_malignant']
  fp = tech_row['cm_true_benign_pred_malignant'] + tech_row['cm_true_normal_pred_malignant']
  fn = tech_row['cm_true_malignant_pred_benign'] + tech_row['cm_true_malignant_pred_normal']
  tn = tech_row['n_images'] - tp - fp - fn
  prevalence = y_true_binary.sum() / len(y_true_binary)

  row = {
      'architecture': architecture, 'mode': mode, 'arm': arm, 'eval_data': eval_data,
      'malignant_fnr': fn / (fn + tp) if (fn + tp) > 0 else np.nan,
      'malignant_ppv': tp / (tp + fp) if (tp + fp) > 0 else np.nan,
      'malignant_npv': tn / (tn + fn) if (tn + fn) > 0 else np.nan,
      'malignant_prevalence': prevalence,
  }
  for target, vals in sens_spec.items():
    pct = int(target * 100)
    row[f'sensitivity_at_spec{pct}'] = vals['sensitivity']
    row[f'threshold_at_spec{pct}'] = vals['threshold']

  clinical_results.append(row)

clinical_df = pd.DataFrame(clinical_results)
print(f'clinical_df: {clinical_df.shape}')

### 5.3 Save on the Disk

FNR cross-checked against `1-recall_malignant`, sensitivity confirmed monotonic across specificity targets, then saved.

In [ ]:
if len(clinical_df) != 54:
  raise ValueError(f'Expected 54 rows, got {len(clinical_df)}.')

# Sensitivity at each target should be non-decreasing as the specificity target loosens.
if not (clinical_df['sensitivity_at_spec95'] <= clinical_df['sensitivity_at_spec90'] + 1e-9).all():
  raise ValueError('sensitivity_at_spec95 > sensitivity_at_spec90 somewhere -- interpolation bug.')
if not (clinical_df['sensitivity_at_spec90'] <= clinical_df['sensitivity_at_spec85'] + 1e-9).all():
  raise ValueError('sensitivity_at_spec90 > sensitivity_at_spec85 somewhere -- interpolation bug.')

if clinical_df[['malignant_ppv', 'malignant_npv']].isna().any().any():
  print('Note: NaN in PPV/NPV -- check for a combination with zero predicted/actual positives/negatives.')

print('Verified: 54 rows, sensitivity monotonic across specificity targets.')

clinical_df.to_csv(EVAL_DIR / 'clinical_metrics.csv', index=False)
print(f'Saved to {EVAL_DIR / "clinical_metrics.csv"}')

## 6. Statistical Significance

### 6.1 Comparison Registry

Enumerates all 171 comparisons:


*   39 confirmatory (inpainted arm: TL-vs-scratch, architecture-vs-architecture,internal-vs-external),
*   132 exploratory (the same three families replicated with annotated/masked, plus arm-vs-arm itself).

Each row identifies two conditions (`_a`/`_b`) plus its family and pairing type: `type1_paired` (same images, McNemar/DeLong/paired-bootstrap all apply) or `type2_unpaired` (different images, unpaired-bootstrap only).



In [ ]:
ARCHITECTURES = ['resnet50', 'efficientnet_b0', 'swin_tiny']
MODES = ['TL', 'scratch']
EXTERNAL_DATASETS = ['bus_uclm', 'breast']
ARCH_PAIRS = list(itertools.combinations(ARCHITECTURES, 2))
ARM_PAIRS = list(itertools.combinations(['annotated', 'inpainted', 'masked'], 2))

registry_rows = []

def add_comparison(comparison_type, family, pairing_type, cond_a, cond_b):
  registry_rows.append({
      'comparison_type': comparison_type, 'family': family, 'pairing_type': pairing_type,
      'arch_a': cond_a[0], 'mode_a': cond_a[1], 'arm_a': cond_a[2], 'eval_data_a': cond_a[3],
      'arch_b': cond_b[0], 'mode_b': cond_b[1], 'arm_b': cond_b[2], 'eval_data_b': cond_b[3],
  })

# --- Confirmatory (inpainted arm only) ---
for arch in ARCHITECTURES:
  for ed in EVAL_DATASETS:
    add_comparison('tl_vs_scratch', 'confirmatory', 'type1_paired',
                    (arch, 'TL', 'inpainted', ed), (arch, 'scratch', 'inpainted', ed))

for mode in MODES:
  for ed in EVAL_DATASETS:
    for a1, a2 in ARCH_PAIRS:
      add_comparison('architecture_vs_architecture', 'confirmatory', 'type1_paired',
                      (a1, mode, 'inpainted', ed), (a2, mode, 'inpainted', ed))

for arch in ARCHITECTURES:
  for mode in MODES:
    for ext in EXTERNAL_DATASETS:
      add_comparison('internal_vs_external', 'confirmatory', 'type2_unpaired',
                      (arch, mode, 'inpainted', 'busi_test'), (arch, mode, 'inpainted', ext))

# --- Exploratory (annotated/masked replication + arm-vs-arm) ---
for arch in ARCHITECTURES:
  for arm in ['annotated', 'masked']:
    for ed in EVAL_DATASETS:
      add_comparison('tl_vs_scratch', 'exploratory', 'type1_paired',
                      (arch, 'TL', arm, ed), (arch, 'scratch', arm, ed))

for mode in MODES:
  for arm in ['annotated', 'masked']:
    for ed in EVAL_DATASETS:
      for a1, a2 in ARCH_PAIRS:
        add_comparison('architecture_vs_architecture', 'exploratory', 'type1_paired',
                        (a1, mode, arm, ed), (a2, mode, arm, ed))

for arch in ARCHITECTURES:
  for mode in MODES:
    for arm in ['annotated', 'masked']:
      for ext in EXTERNAL_DATASETS:
        add_comparison('internal_vs_external', 'exploratory', 'type2_unpaired',
                        (arch, mode, arm, 'busi_test'), (arch, mode, arm, ext))

for arch in ARCHITECTURES:
  for mode in MODES:
    for ed in EVAL_DATASETS:
      for arm1, arm2 in ARM_PAIRS:
        add_comparison('arm_vs_arm', 'exploratory', 'type1_paired',
                        (arch, mode, arm1, ed), (arch, mode, arm2, ed))

comparison_registry = pd.DataFrame(registry_rows)
comparison_registry.insert(0, 'comparison_id', range(len(comparison_registry)))
print(f'comparison_registry: {len(comparison_registry)} rows')

### 6.2 McNemar's Test

Paired test on binary correct/incorrect outcomes. Automatically switches to the exact (binomial) form when discordant pairs number fewer than 25, and the chi-square form with continuity correction otherwise, considering that chi-square approximation is unreliable on small discordant counts, a real risk considering some `type1_paired` comparisons involve datasets as small as BUSI test (n=80).

In [ ]:
def mcnemar_test(correct_a, correct_b, exact_threshold=25):
  correct_a = np.asarray(correct_a, dtype=bool)
  correct_b = np.asarray(correct_b, dtype=bool)
  if len(correct_a) != len(correct_b):
    raise ValueError('Arrays must be the same length (paired).')

  b = int(np.sum(correct_a & ~correct_b))   # A right, B wrong
  c = int(np.sum(~correct_a & correct_b))   # A wrong, B right
  n_discordant = b + c

  if n_discordant == 0:
    return {'b': b, 'c': c, 'n_discordant': 0, 'method': 'undefined', 'statistic': None, 'p_value': 1.0}

  if n_discordant < exact_threshold:
    k = min(b, c)
    p_value = scipy_stats.binomtest(k, n_discordant, 0.5, alternative='two-sided').pvalue
    return {'b': b, 'c': c, 'n_discordant': n_discordant, 'method': 'exact', 'statistic': None, 'p_value': p_value}
  else:
    chi2_stat = (abs(b - c) - 1) ** 2 / n_discordant
    p_value = scipy_stats.chi2.sf(chi2_stat, df=1)
    return {'b': b, 'c': c, 'n_discordant': n_discordant, 'method': 'chi2_corrected', 'statistic': chi2_stat, 'p_value': p_value}

In [ ]:
# Case 1: small discordant count (b=2, c=0) -- should trigger 'exact'.
correct_a_small = [1,1,1,1,1,0,0,0,1,1]
correct_b_small = [1,1,1,0,0,0,0,0,1,1]
result_small = mcnemar_test(correct_a_small, correct_b_small)

# Independent check: raw two-sided binomial-sum p-value, not via scipy's binomtest.
n, k = result_small['n_discordant'], min(result_small['b'], result_small['c'])
independent_exact_p = min(1.0, 2 * sum(math.comb(n, i) * 0.5**n for i in range(0, k + 1)))

# Case 2: large discordant count (b=10, c=30) -- should trigger chi-square.
correct_a_large = np.array([True]*10 + [False]*30 + [True]*10)
correct_b_large = np.array([False]*10 + [True]*30 + [True]*10)
result_large = mcnemar_test(correct_a_large, correct_b_large)

# Independent check: chi-square arithmetic computed separately from the function.
b2, c2 = 10, 30
independent_chi2_stat = (abs(b2 - c2) - 1) ** 2 / (b2 + c2)
independent_chi2_p = scipy_stats.chi2.sf(independent_chi2_stat, df=1)

checks = [
    ("Case 1 selects 'exact'", result_small['method'] == 'exact'),
    ('Case 1 p-value matches independent binomial sum', abs(result_small['p_value'] - independent_exact_p) < 1e-9),
    ("Case 2 selects 'chi2_corrected'", result_large['method'] == 'chi2_corrected'),
    ('Case 2 statistic matches independent arithmetic', abs(result_large['statistic'] - independent_chi2_stat) < 1e-9),
    ('Case 2 p-value matches independent arithmetic', abs(result_large['p_value'] - independent_chi2_p) < 1e-9),
]

all_passed = True
for description, passed in checks:
  print(f'[{"PASS" if passed else "FAIL"}] {description}')
  if not passed:
    all_passed = False

if not all_passed:
  raise ValueError('McNemar verification failed.')
print('\nmcnemar_test verified against two independently-computed reference cases.')

### 6.3 DeLong's Test

Built directly from the covariance-based formula (DeLong et al. 1988; Sun & Xu 2014). Verified against two independently published worked examples (Draelos 2020), plus a paired-bootstrap cross-check of the standard error as a second, independent verification method. Per-class only.

In [ ]:
def _psi_matrix(X, Y):
  """psi(x,y) = 1 if y<x, 0.5 if y==x, 0 if y>x -- reward when the positive-class
  score outranks the negative-class score. X: positive scores (m,), Y: negative
  scores (n,). Returns the full (m, n) comparison matrix via broadcasting."""
  diff = X[:, None] - Y[None, :]
  return np.where(diff > 0, 1.0, np.where(diff == 0, 0.5, 0.0))


def delong_test(y_true_binary, score_a, score_b):
  """DeLong's test for two correlated AUCs on the same binary outcome, same cases.
  Returns AUCs, z-statistic, and two-sided p-value."""
  y_true_binary = np.asarray(y_true_binary)
  score_a = np.asarray(score_a, dtype=float)
  score_b = np.asarray(score_b, dtype=float)

  Xa, Ya = score_a[y_true_binary == 1], score_a[y_true_binary == 0]
  Xb, Yb = score_b[y_true_binary == 1], score_b[y_true_binary == 0]
  m, n = len(Xa), len(Ya)

  psi_a, psi_b = _psi_matrix(Xa, Ya), _psi_matrix(Xb, Yb)
  V10_a, V01_a = psi_a.mean(axis=1), psi_a.mean(axis=0)
  V10_b, V01_b = psi_b.mean(axis=1), psi_b.mean(axis=0)

  auc_a, auc_b = V10_a.mean(), V10_b.mean()

  S10 = np.cov(np.stack([V10_a, V10_b], axis=1), rowvar=False, ddof=1)
  S01 = np.cov(np.stack([V01_a, V01_b], axis=1), rowvar=False, ddof=1)
  S = S10 / m + S01 / n

  var_diff = S[0, 0] + S[1, 1] - 2 * S[0, 1]
  z = (auc_a - auc_b) / np.sqrt(var_diff)
  p_value = 2 * (1 - scipy_stats.norm.cdf(abs(z)))

  return {'auc_a': auc_a, 'auc_b': auc_b, 'z': z, 'p_value': p_value}

In [ ]:
# Both examples from Draelos (2020), hand-derived by the author and independently
# cross-checked against R's pROC package.
example_1 = {
    'y_true': [0, 0, 1, 1, 1], 'score_a': [0.1, 0.2, 0.6, 0.7, 0.8], 'score_b': [0.3, 0.6, 0.2, 0.7, 0.9],
    'expected': {'auc_a': 1.0, 'auc_b': 0.6666667, 'z': 1.0, 'p_value': 0.3173},
}
example_2 = {
    'y_true': [0,0,0,0,0,0,1,1,1,1,1,1,1],
    'score_a': [0.1,0.2,0.05,0.3,0.1,0.6,0.6,0.7,0.8,0.99,0.8,0.67,0.5],
    'score_b': [0.3,0.6,0.2,0.1,0.1,0.9,0.23,0.7,0.9,0.4,0.77,0.3,0.89],
    'expected': {'auc_a': 0.9642857, 'auc_b': 0.7380952, 'z': 1.672, 'p_value': 0.09453},
}

all_passed = True
for name, ex in [('Example 1', example_1), ('Example 2', example_2)]:
  result = delong_test(ex['y_true'], ex['score_a'], ex['score_b'])
  for key, expected_val in ex['expected'].items():
    tol = 1e-3 if key != 'p_value' else 2e-3
    if abs(result[key] - expected_val) >= tol:
      all_passed = False
      print(f'[FAIL] {name} {key}: expected {expected_val}, got {result[key]:.4f}')

if all_passed:
  print('DeLong implementation matches both published worked examples exactly.')
else:
  raise ValueError('DeLong verification failed against published examples.')

In [ ]:
np.random.seed(SEED)
y_true_arr = np.array(example_2['y_true'])
score_a_arr = np.array(example_2['score_a'])
score_b_arr = np.array(example_2['score_b'])
n_boot = 2000

diffs = []
for _ in range(n_boot):
  sample = np.random.choice(len(y_true_arr), size=len(y_true_arr), replace=True)
  yt = y_true_arr[sample]
  if len(np.unique(yt)) < 2:
    continue
  diffs.append(roc_auc_score(yt, score_a_arr[sample]) - roc_auc_score(yt, score_b_arr[sample]))

bootstrap_se = np.std(diffs, ddof=1)
delong_result = delong_test(example_2['y_true'], example_2['score_a'], example_2['score_b'])
delong_se = (delong_result['auc_a'] - delong_result['auc_b']) / delong_result['z']
ratio = bootstrap_se / delong_se

print(f'DeLong analytical SE: {delong_se:.4f}')
print(f'Bootstrap empirical SE ({len(diffs)} resamples): {bootstrap_se:.4f}')
print(f'Ratio: {ratio:.3f}')

if not (0.7 <= ratio <= 1.3):
  raise ValueError(f'DeLong SE and bootstrap SE disagree substantially (ratio={ratio:.3f}) -- investigate before trusting this function.')
print('\nTwo independent verification methods agree: published-example match and bootstrap SE cross-check.')

### 6.4 Bootstrap Confidence Intervals

Stratified resampling. Paired bootstrap for Type 1 comparisons, preserving correlated errors on hard images. Unpaired bootstrap for Type 2 comparisons, for internal-vs-external comparisons. Covers the following three metrics: macro-F1, malignant sensitivity at 90% specificity, macro-AUC.

In [ ]:
def stratified_resample_indices(y_true, rng):
  """One stratified resample: each class resampled with replacement from its
  own pool, at its own original size. Guarantees every class is represented
  in every resample, regardless of how rare it is."""
  y_true = np.asarray(y_true)
  resampled = []
  for cls in range(3):
    cls_idx = np.where(y_true == cls)[0]
    if len(cls_idx) == 0:
      continue
    resampled.append(rng.choice(cls_idx, size=len(cls_idx), replace=True))
  return np.concatenate(resampled)


def metric_macro_f1(y_true, pred_label):
  return f1_score(y_true, pred_label, average='macro', zero_division=0, labels=[0, 1, 2])


def metric_malignant_sensitivity_90(y_true, y_prob):
  y_true_binary = (y_true == MALIGNANT_IDX).astype(int)
  return compute_sensitivity_at_specificity(y_true_binary, y_prob[:, MALIGNANT_IDX], targets=[0.90])[0.90]['sensitivity']


def metric_macro_auc(y_true, y_prob):
  aucs = [roc_auc_score((y_true == i).astype(int), y_prob[:, i]) for i in range(3)]
  return np.mean(aucs)


def paired_bootstrap_diff(y_true, values_a, values_b, metric_fn, n_boot=10000, seed=SEED):
  """Type 1 only -- same resampled indices applied to both A and B each
  iteration, preserving correlation between their errors on the same images."""
  rng = np.random.default_rng(seed)
  y_true, values_a, values_b = np.asarray(y_true), np.asarray(values_a), np.asarray(values_b)
  point_diff = metric_fn(y_true, values_a) - metric_fn(y_true, values_b)

  diffs = np.empty(n_boot)
  for i in range(n_boot):
    idx = stratified_resample_indices(y_true, rng)
    diffs[i] = metric_fn(y_true[idx], values_a[idx]) - metric_fn(y_true[idx], values_b[idx])

  p_value = min(2 * min((diffs >= 0).mean(), (diffs <= 0).mean()), 1.0)
  p_value = max(p_value, 1 / n_boot)  # floor: a bootstrap p-value can't resolve below ~1/n_boot
  return {'point_estimate': point_diff, 'ci_lower': np.percentile(diffs, 2.5),
          'ci_upper': np.percentile(diffs, 97.5), 'p_value': p_value}


def unpaired_bootstrap_diff(y_true_a, values_a, y_true_b, values_b, metric_fn, n_boot=10000, seed=SEED):
  """Type 2 only: internal-vs-external, different images entirely. Each
  dataset resampled independently; pairing is impossible, not applicable."""
  rng = np.random.default_rng(seed)
  y_true_a, values_a = np.asarray(y_true_a), np.asarray(values_a)
  y_true_b, values_b = np.asarray(y_true_b), np.asarray(values_b)
  point_diff = metric_fn(y_true_a, values_a) - metric_fn(y_true_b, values_b)

  diffs = np.empty(n_boot)
  for i in range(n_boot):
    idx_a = stratified_resample_indices(y_true_a, rng)
    idx_b = stratified_resample_indices(y_true_b, rng)
    diffs[i] = metric_fn(y_true_a[idx_a], values_a[idx_a]) - metric_fn(y_true_b[idx_b], values_b[idx_b])

  p_value = min(2 * min((diffs >= 0).mean(), (diffs <= 0).mean()), 1.0)
  p_value = max(p_value, 1 / n_boot)
  return {'point_estimate': point_diff, 'ci_lower': np.percentile(diffs, 2.5),
          'ci_upper': np.percentile(diffs, 97.5), 'p_value': p_value}

In [ ]:
rng_check = np.random.default_rng(0)
y_true_imbalanced = np.array([0]*30 + [1]*10 + [2]*4)  # mimics BrEaST-scale imbalance
counts_preserved = all(
    [np.sum(y_true_imbalanced[stratified_resample_indices(y_true_imbalanced, rng_check)] == c) for c in range(3)] == [30, 10, 4]
    for _ in range(500)
)

np.random.seed(1)
n = 80
y_true_a = np.random.choice([0, 1, 2], size=n, p=[0.5, 0.3, 0.2])
pred_a = np.where(np.random.rand(n) < 0.9, y_true_a, np.random.choice([0, 1, 2], size=n))
pred_b = np.random.choice([0, 1, 2], size=n)  # near-random -- clearly worse

result_diff = paired_bootstrap_diff(y_true_a, pred_a, pred_b, metric_macro_f1, n_boot=3000)
result_same = paired_bootstrap_diff(y_true_a, pred_a, pred_a, metric_macro_f1, n_boot=3000)

y_true_b = np.random.choice([0, 1, 2], size=60, p=[0.6, 0.1, 0.3])
pred_b2 = np.random.choice([0, 1, 2], size=60)
result_unpaired = unpaired_bootstrap_diff(y_true_a, pred_a, y_true_b, pred_b2, metric_macro_f1, n_boot=3000)

checks = [
    ('Stratified resampling preserves exact per-class counts, 500/500 trials', counts_preserved),
    ('Clearly-different paired models: CI excludes 0', not (result_diff['ci_lower'] <= 0 <= result_diff['ci_upper'])),
    ('Clearly-different paired models: p < 0.05', result_diff['p_value'] < 0.05),
    ('Identical models: point estimate exactly 0', result_same['point_estimate'] == 0),
    ('Identical models: every resample gives diff=0, so CI = [0, 0]', result_same['ci_lower'] == 0 and result_same['ci_upper'] == 0),
    ('Unpaired, clearly different: CI excludes 0', not (result_unpaired['ci_lower'] <= 0 <= result_unpaired['ci_upper'])),
]

all_passed = True
for description, passed in checks:
  print(f'[{"PASS" if passed else "FAIL"}] {description}')
  if not passed:
    all_passed = False

if not all_passed:
  raise ValueError('Bootstrap verification failed.')
print('\nBootstrap engine verified against constructed cases with known expected behavior.')

### 6.5 Apply All Applicable Tests Across the Registry

Checkpointed by design: the results are appended to disk after every single comparison, not just at the end of a batch. If a session disconnects mid-run, re-running the exact same cell picks up where it left off.

In [ ]:
import time

STATS_RESULTS_PATH = EVAL_DIR / 'statistical_results_raw.csv'


def get_predictions_for_condition(architecture, mode, arm, eval_data):
  group = predictions_df[(predictions_df['architecture'] == architecture) & (predictions_df['mode'] == mode) &
                          (predictions_df['arm'] == arm) & (predictions_df['eval_data'] == eval_data)]
  return group.sort_values('image_name').reset_index(drop=True)


def run_comparison(row):
  cond_a = get_predictions_for_condition(row['arch_a'], row['mode_a'], row['arm_a'], row['eval_data_a'])
  cond_b = get_predictions_for_condition(row['arch_b'], row['mode_b'], row['arm_b'], row['eval_data_b'])
  result = {
      'comparison_id': row['comparison_id'], 'comparison_type': row['comparison_type'],
      'family': row['family'], 'pairing_type': row['pairing_type'],
      'arch_a': row['arch_a'], 'mode_a': row['mode_a'], 'arm_a': row['arm_a'], 'eval_data_a': row['eval_data_a'],
      'arch_b': row['arch_b'], 'mode_b': row['mode_b'], 'arm_b': row['arm_b'], 'eval_data_b': row['eval_data_b'],
  }

  if row['pairing_type'] == 'type1_paired':
    if cond_a['image_name'].tolist() != cond_b['image_name'].tolist():
      raise ValueError(f"comparison_id {row['comparison_id']}: image_name mismatch between paired conditions.")

    y_true = cond_a['true_label'].values
    pred_a, pred_b = cond_a['pred_label'].values, cond_b['pred_label'].values
    prob_a = cond_a[['prob_benign', 'prob_malignant', 'prob_normal']].values
    prob_b = cond_b[['prob_benign', 'prob_malignant', 'prob_normal']].values

    mcnemar_result = mcnemar_test(pred_a == y_true, pred_b == y_true)
    result['mcnemar_p'] = mcnemar_result['p_value']
    result['mcnemar_method'] = mcnemar_result['method']

    for i, cls in enumerate(CLASS_NAMES):
      y_true_binary = (y_true == i).astype(int)
      if len(np.unique(y_true_binary)) < 2:
        result[f'delong_{cls}_p'] = np.nan
      else:
        result[f'delong_{cls}_p'] = delong_test(y_true_binary, prob_a[:, i], prob_b[:, i])['p_value']

    boot_f1 = paired_bootstrap_diff(y_true, pred_a, pred_b, metric_macro_f1, n_boot=10000)
    boot_sens = paired_bootstrap_diff(y_true, prob_a, prob_b, metric_malignant_sensitivity_90, n_boot=10000)
    boot_auc = paired_bootstrap_diff(y_true, prob_a, prob_b, metric_macro_auc, n_boot=10000)

  else:  # type2_unpaired
    y_true_a, y_true_b = cond_a['true_label'].values, cond_b['true_label'].values
    pred_a, pred_b = cond_a['pred_label'].values, cond_b['pred_label'].values
    prob_a = cond_a[['prob_benign', 'prob_malignant', 'prob_normal']].values
    prob_b = cond_b[['prob_benign', 'prob_malignant', 'prob_normal']].values

    result['mcnemar_p'] = np.nan
    result['mcnemar_method'] = 'not_applicable'
    for cls in CLASS_NAMES:
      result[f'delong_{cls}_p'] = np.nan

    boot_f1 = unpaired_bootstrap_diff(y_true_a, pred_a, y_true_b, pred_b, metric_macro_f1, n_boot=10000)
    boot_sens = unpaired_bootstrap_diff(y_true_a, prob_a, y_true_b, prob_b, metric_malignant_sensitivity_90, n_boot=10000)
    boot_auc = unpaired_bootstrap_diff(y_true_a, prob_a, y_true_b, prob_b, metric_macro_auc, n_boot=10000)

  for metric_name, boot in [('macro_f1', boot_f1), ('sensitivity90', boot_sens), ('macro_auc', boot_auc)]:
    result[f'bootstrap_{metric_name}_diff'] = boot['point_estimate']
    result[f'bootstrap_{metric_name}_ci_lower'] = boot['ci_lower']
    result[f'bootstrap_{metric_name}_ci_upper'] = boot['ci_upper']
    result[f'bootstrap_{metric_name}_p'] = boot['p_value']

  return result


def run_comparisons_batch(comparison_ids, results_path=STATS_RESULTS_PATH):
  done_ids = set(pd.read_csv(results_path)['comparison_id']) if results_path.exists() else set()
  remaining = [cid for cid in comparison_ids if cid not in done_ids]
  print(f'{len(remaining)} of {len(comparison_ids)} remaining ({len(comparison_ids) - len(remaining)} already done).')

  for i, cid in enumerate(remaining):
    row = comparison_registry[comparison_registry['comparison_id'] == cid].iloc[0]
    t0 = time.perf_counter()
    result = run_comparison(row)
    elapsed = time.perf_counter() - t0

    pd.DataFrame([result]).to_csv(results_path, mode='a', header=not results_path.exists(), index=False)
    print(f'[{i+1}/{len(remaining)}] id={cid} ({row["comparison_type"]}, {row["family"]}) -- {elapsed:.1f}s')

  print('\nBatch complete.')

In [ ]:
# Confirmatory Type 1 (27 comparisons)
ids = comparison_registry[(comparison_registry['family'] == 'confirmatory') &
                           (comparison_registry['pairing_type'] == 'type1_paired')]['comparison_id'].tolist()
run_comparisons_batch(ids)

In [ ]:
# Confirmatory Type 2 (12 comparisons)
ids = comparison_registry[(comparison_registry['family'] == 'confirmatory') &
                           (comparison_registry['pairing_type'] == 'type2_unpaired')]['comparison_id'].tolist()
run_comparisons_batch(ids)

In [ ]:
# Exploratory Type 1 (108 comparisons)
ids = comparison_registry[(comparison_registry['family'] == 'exploratory') &
                           (comparison_registry['pairing_type'] == 'type1_paired')]['comparison_id'].tolist()
run_comparisons_batch(ids)

In [ ]:
# Exploratory Type 2 (24 comparisons)
ids = comparison_registry[(comparison_registry['family'] == 'exploratory') &
                           (comparison_registry['pairing_type'] == 'type2_unpaired')]['comparison_id'].tolist()
run_comparisons_batch(ids)

In [ ]:
stats_raw = pd.read_csv(STATS_RESULTS_PATH)

checks = []
checks.append(('171 rows total', len(stats_raw) == 171))
checks.append(('No duplicate comparison_ids', stats_raw['comparison_id'].duplicated().sum() == 0))
checks.append(('Every registry comparison_id present',
                set(stats_raw['comparison_id']) == set(comparison_registry['comparison_id'])))

merged = stats_raw
type1 = merged[merged['pairing_type'] == 'type1_paired']
type2 = merged[merged['pairing_type'] == 'type2_unpaired']

mcnemar_delong_cols = ['mcnemar_p', 'delong_benign_p', 'delong_malignant_p', 'delong_normal_p']
checks.append(('Type1 rows: no unexpected NaN in McNemar/DeLong', not type1[mcnemar_delong_cols].isna().any().any()))
checks.append(('Type2 rows: McNemar/DeLong all NaN (not applicable)', type2[mcnemar_delong_cols].isna().all().all()))

boot_cols = [c for c in stats_raw.columns if c.startswith('bootstrap_')]
checks.append(('No NaN in any bootstrap column, any row', not merged[boot_cols].isna().any().any()))

p_cols = [c for c in stats_raw.columns if c.endswith('_p')]
all_p = pd.concat([merged[c].dropna() for c in p_cols])
checks.append(('All p-values in [0, 1]', all_p.between(0, 1).all()))

spot = comparison_registry[(comparison_registry['arch_a'] == 'resnet50') & (comparison_registry['mode_a'] == 'TL') &
                            (comparison_registry['arm_a'] == 'inpainted') & (comparison_registry['eval_data_a'] == 'busi_test') &
                            (comparison_registry['comparison_type'] == 'tl_vs_scratch')].iloc[0]
spot_result = stats_raw[stats_raw['comparison_id'] == spot['comparison_id']].iloc[0]
f1_a = technical_df[(technical_df['architecture']=='resnet50')&(technical_df['mode']=='TL')&(technical_df['arm']=='inpainted')&(technical_df['eval_data']=='busi_test')]['macro_f1'].iloc[0]
f1_b = technical_df[(technical_df['architecture']=='resnet50')&(technical_df['mode']=='scratch')&(technical_df['arm']=='inpainted')&(technical_df['eval_data']=='busi_test')]['macro_f1'].iloc[0]
checks.append(('Spot-check: bootstrap point estimate matches technical_df macro_f1 diff',
                abs(spot_result['bootstrap_macro_f1_diff'] - (f1_a - f1_b)) < 1e-9))

all_passed = True
for description, passed in checks:
  print(f'[{"PASS" if passed else "FAIL"}] {description}')
  if not passed:
    all_passed = False

if not all_passed:
  raise ValueError('6.5 verification failed.')
print('\n6.5 verified: 171 comparisons, correctly structured, spot-check matches Section 4.')

### 6.6 Multiple-Comparison Correction

Holm-Bonferroni within each confirmatory test-type, Benjamini-Hochberg within each exploratory test-type; 14 separate correction groups total, per the granularity split (McNemar, DeLong × 3 classes, bootstrap × 3 metrics, × 2 families).

Both are built directly and verified against a published, software-tested reference `adjustp`.

In [ ]:
def holm_bonferroni(p_values, alpha=0.05):
  p_values = np.asarray(p_values, dtype=float)
  m = len(p_values)
  order = np.argsort(p_values)
  sorted_p = p_values[order]
  terms = np.minimum(1.0, (m - np.arange(m)) * sorted_p)
  adjusted_sorted = np.maximum.accumulate(terms)
  adjusted = np.empty(m)
  adjusted[order] = adjusted_sorted
  return adjusted, adjusted < alpha


def benjamini_hochberg(p_values, alpha=0.05):
  p_values = np.asarray(p_values, dtype=float)
  m = len(p_values)
  order = np.argsort(p_values)
  sorted_p = p_values[order]
  ranks = np.arange(1, m + 1)
  terms = np.minimum(1.0, (m / ranks) * sorted_p)
  adjusted_sorted = np.minimum.accumulate(terms[::-1])[::-1]
  adjusted = np.empty(m)
  adjusted[order] = adjusted_sorted
  return adjusted, adjusted < alpha

In [ ]:
# Published reference (adjustp, Rust crate, doctested), independently hand-verified beforehand.
p_ref = [0.1, 0.2, 0.3, 0.4, 0.1]
expected_bh_ref = [0.25, 0.33333333333333337, 0.375, 0.4, 0.25]
expected_holm_hand = [0.5, 0.6, 0.6, 0.6, 0.5]  # hand-derived, same test vector

bh_adj, _ = benjamini_hochberg(p_ref)
holm_adj, _ = holm_bonferroni(p_ref)

checks = [
    ('BH matches published reference', np.allclose(bh_adj, expected_bh_ref)),
    ('Holm matches hand-derivation', np.allclose(holm_adj, expected_holm_hand)),
]

rng_check = np.random.default_rng(3)
for m_test in [27, 39, 108, 132]:  # actual group sizes used below
  p_rand = rng_check.beta(0.5, 5, size=m_test)
  h_adj, _ = holm_bonferroni(p_rand)
  b_adj, _ = benjamini_hochberg(p_rand)
  checks.append((f'Holm adjusted >= raw p, m={m_test}', np.all(h_adj >= p_rand - 1e-12)))
  checks.append((f'BH adjusted >= raw p, m={m_test}', np.all(b_adj >= p_rand - 1e-12)))

all_passed = True
for description, passed in checks:
  print(f'[{"PASS" if passed else "FAIL"}] {description}')
  if not passed:
    all_passed = False

if not all_passed:
  raise ValueError('Correction function verification failed.')
print('\nHolm-Bonferroni and Benjamini-Hochberg verified.')

In [ ]:
CORRECTION_GROUPS = [
    ('mcnemar_p', 'confirmatory', 'type1_paired'), ('delong_benign_p', 'confirmatory', 'type1_paired'),
    ('delong_malignant_p', 'confirmatory', 'type1_paired'), ('delong_normal_p', 'confirmatory', 'type1_paired'),
    ('bootstrap_macro_f1_p', 'confirmatory', None), ('bootstrap_sensitivity90_p', 'confirmatory', None),
    ('bootstrap_macro_auc_p', 'confirmatory', None),
    ('mcnemar_p', 'exploratory', 'type1_paired'), ('delong_benign_p', 'exploratory', 'type1_paired'),
    ('delong_malignant_p', 'exploratory', 'type1_paired'), ('delong_normal_p', 'exploratory', 'type1_paired'),
    ('bootstrap_macro_f1_p', 'exploratory', None), ('bootstrap_sensitivity90_p', 'exploratory', None),
    ('bootstrap_macro_auc_p', 'exploratory', None),
]
P_COLUMNS = ['mcnemar_p', 'delong_benign_p', 'delong_malignant_p', 'delong_normal_p',
             'bootstrap_macro_f1_p', 'bootstrap_sensitivity90_p', 'bootstrap_macro_auc_p']

stats_corrected = stats_raw.copy()
for col in P_COLUMNS:
  stats_corrected[f'{col}_adj'] = np.nan
  stats_corrected[f'{col}_significant'] = False

group_sizes = {}
for p_col, family, pairing_filter in CORRECTION_GROUPS:
  mask = (stats_corrected['family'] == family) & stats_corrected[p_col].notna()
  if pairing_filter is not None:
    mask &= stats_corrected['pairing_type'] == pairing_filter

  p_vals = stats_corrected.loc[mask, p_col].values
  method = 'holm' if family == 'confirmatory' else 'BH'
  adj, sig = (holm_bonferroni(p_vals) if method == 'holm' else benjamini_hochberg(p_vals))

  stats_corrected.loc[mask, f'{p_col}_adj'] = adj
  stats_corrected.loc[mask, f'{p_col}_significant'] = sig
  group_sizes[(p_col, family)] = (method, mask.sum())

for k, (method, n) in group_sizes.items():
  print(f'{k}: method={method}, n={n}')

In [ ]:
expected_sizes = {
    ('mcnemar_p', 'confirmatory'): 27, ('delong_benign_p', 'confirmatory'): 27,
    ('delong_malignant_p', 'confirmatory'): 27, ('delong_normal_p', 'confirmatory'): 27,
    ('bootstrap_macro_f1_p', 'confirmatory'): 39, ('bootstrap_sensitivity90_p', 'confirmatory'): 39,
    ('bootstrap_macro_auc_p', 'confirmatory'): 39,
    ('mcnemar_p', 'exploratory'): 108, ('delong_benign_p', 'exploratory'): 108,
    ('delong_malignant_p', 'exploratory'): 108, ('delong_normal_p', 'exploratory'): 108,
    ('bootstrap_macro_f1_p', 'exploratory'): 132, ('bootstrap_sensitivity90_p', 'exploratory'): 132,
    ('bootstrap_macro_auc_p', 'exploratory'): 132,
}

checks = []
for k, expected_n in expected_sizes.items():
  checks.append((f'{k} group size', group_sizes[k][1] == expected_n))

adj_cols = [f'{c}_adj' for c in P_COLUMNS]
raw_adj_pairs = [(c, f'{c}_adj') for c in P_COLUMNS]
for raw_col, adj_col in raw_adj_pairs:
  valid = stats_corrected[raw_col].notna()
  checks.append((f'{adj_col} >= {raw_col} everywhere valid',
                  (stats_corrected.loc[valid, adj_col] >= stats_corrected.loc[valid, raw_col] - 1e-9).all()))
  checks.append((f'{adj_col} in [0,1]', stats_corrected.loc[valid, adj_col].between(0, 1).all()))

all_passed = True
for description, passed in checks:
  print(f'[{"PASS" if passed else "FAIL"}] {description}')
  if not passed:
    all_passed = False

if not all_passed:
  raise ValueError('6.6 verification failed.')
print('\n6.6 verified: all 14 correction groups correctly sized, adjusted p-values valid.')

#### 6.6.1 Save Corrected Results

`stats_corrected` was built in-memory by applying Holm/BH correction to `statistical_results_raw.csv`. Saved here so the corrected p-values and significance flags are independently inspectable.

In [ ]:
stats_corrected.to_csv(EVAL_DIR / 'statistical_results_corrected.csv', index=False)

reloaded = pd.read_csv(EVAL_DIR / 'statistical_results_corrected.csv')
if reloaded.shape != stats_corrected.shape:
  raise ValueError(f'Shape mismatch: saved {reloaded.shape}, in-memory {stats_corrected.shape}')

adj_cols = [c for c in stats_corrected.columns if c.endswith('_adj')]
sig_cols = [c for c in stats_corrected.columns if c.endswith('_significant')]

# NaN == NaN is always False -- use allclose for numeric adj columns
# (tolerates float precision loss in CSV round-trip) and
# fillna for boolean significant columns (NaN treated as a third
# distinct value, not a mismatch).
for col in adj_cols:
  orig = stats_corrected[col].values.astype(float)
  saved = reloaded[col].values.astype(float)
  if not np.allclose(orig, saved, equal_nan=True):
    raise ValueError(f'Adj column {col} differs after round-trip.')

for col in sig_cols:
  orig = stats_corrected[col].fillna(-1)
  saved = reloaded[col].fillna(-1)
  if not (orig == saved).all():
    raise ValueError(f'Significant column {col} differs after round-trip.')

print(f'Saved: {EVAL_DIR / "statistical_results_corrected.csv"}')
print(f'{len(stats_corrected)} rows, {len(adj_cols)} adj columns + {len(sig_cols)} significant columns verified.')

## 7. Best-Model Determination

Three-tier evaluation for the choice of the best model.

### 7.1 Tier 0: Safety Floor

Candidate pool starts as all 6 inpainted-arm models (3 architectures × {TL, scratch}), the arm where confirmatory, corrected evidence exists.

Malignant FNR <= 15%, worst across all three datasets: the generalisation-focused reading.

In [ ]:
tl_vs_scratch_ids = comparison_registry[
    (comparison_registry['comparison_type'] == 'tl_vs_scratch') &
    (comparison_registry['family'] == 'confirmatory')
]['comparison_id'].tolist()

tl_check = stats_corrected[stats_corrected['comparison_id'].isin(tl_vs_scratch_ids)].copy()
print(f'{len(tl_check)} confirmatory TL-vs-scratch comparisons (expected 9).')

display_cols = ['arch_a', 'eval_data_a', 'bootstrap_macro_f1_diff',
                'bootstrap_macro_f1_p_adj', 'bootstrap_macro_f1_p_significant']
print(tl_check[display_cols].sort_values(['arch_a', 'eval_data_a']).to_string(index=False))

n_significant = tl_check['bootstrap_macro_f1_p_significant'].sum()
n_tl_favoring = (tl_check['bootstrap_macro_f1_diff'] > 0).sum()
print(f'\n{n_significant}/9 significant after Holm correction, {n_tl_favoring}/9 favoring TL in direction.')

# 4/9 significant (efficientnet_b0 and swin_tiny on external datasets),
# not 9/9 -- TL-only narrowing is not supported. Both modes stay in the
# candidate pool. Arm-scoping to inpainted stays (independently justified).

In [ ]:
candidate_models = [(arch, mode) for arch in ['resnet50', 'efficientnet_b0', 'swin_tiny']
                     for mode in ['TL', 'scratch']]
print(f'{len(candidate_models)} candidates (all modes, inpainted arm only):')
for c in candidate_models:
  print(f'  {c}')

In [ ]:
tier0_results = []
for architecture, mode in candidate_models:
  model_clinical = clinical_df[(clinical_df['architecture'] == architecture) &
                                (clinical_df['mode'] == mode) & (clinical_df['arm'] == 'inpainted')]
  worst_row = model_clinical.loc[model_clinical['malignant_fnr'].idxmax()]
  tier0_results.append({
      'architecture': architecture, 'mode': mode, 'arm': 'inpainted',
      'worst_malignant_fnr': worst_row['malignant_fnr'], 'worst_fnr_eval_data': worst_row['eval_data'],
      'passes_tier0': worst_row['malignant_fnr'] <= 0.15,
  })

tier0_df = pd.DataFrame(tier0_results)
print(tier0_df.to_string(index=False))

if tier0_df['worst_malignant_fnr'].isna().any():
  raise ValueError('NaN in worst_malignant_fnr -- genuine data problem, investigate before proceeding.')

surviving_models = tier0_df[tier0_df['passes_tier0']][['architecture', 'mode']].apply(tuple, axis=1).tolist()
n_passing = len(surviving_models)

if n_passing == 0:
  print(f'\n0/{len(candidate_models)} candidates pass the pre-specified Tier 0 safety floor.')
  print('Confirmed real finding (RQ2), not a pipeline error. Proceeding to Tiers 1-2 across the full')
  print('candidate set -- every result downstream is explicitly labeled as not meeting the safety floor,')
  print('never presented as an unqualified "best model."')
else:
  print(f'\n{n_passing}/{len(candidate_models)} pass Tier 0: {surviving_models}')

In [ ]:
# Check whether the results trend in the same direction for both external validation datasets
fnr_breakdown = clinical_df[clinical_df['arm'] == 'inpainted'].pivot_table(
    index=['architecture', 'mode'], columns='eval_data', values='malignant_fnr'
)
print(fnr_breakdown.to_string())

### 7.2 Tier 1: External Generalisation Performance

Ranks by actual external performance. Two separate rankings: best TL, best scratch, each using only the architecture-vs-architecture comparisons. This cell does not auto-declare a winner.

In [ ]:
def get_external_metrics(architecture, mode):
  clinical_rows = clinical_df[(clinical_df['architecture'] == architecture) & (clinical_df['mode'] == mode) &
                               (clinical_df['arm'] == 'inpainted') & (clinical_df['eval_data'].isin(['bus_uclm', 'breast']))]
  technical_rows = technical_df[(technical_df['architecture'] == architecture) & (technical_df['mode'] == mode) &
                                 (technical_df['arm'] == 'inpainted') & (technical_df['eval_data'].isin(['bus_uclm', 'breast']))]
  return {
      'avg_external_sensitivity90': clinical_rows['sensitivity_at_spec90'].mean(),
      'avg_external_macro_f1': technical_rows['macro_f1'].mean(),
  }

tier1_rows = []
for architecture in ['resnet50', 'efficientnet_b0', 'swin_tiny']:
  for mode in ['TL', 'scratch']:
    metrics = get_external_metrics(architecture, mode)
    tier1_rows.append({'architecture': architecture, 'mode': mode, **metrics})

tier1_df = pd.DataFrame(tier1_rows).sort_values(['mode', 'avg_external_sensitivity90'], ascending=[True, False])
print(tier1_df.to_string(index=False))

In [ ]:
def find_arch_comparison(arch1, arch2, mode, eval_data):
  match = comparison_registry[(comparison_registry['comparison_type'] == 'architecture_vs_architecture') &
                               (comparison_registry['family'] == 'confirmatory') &
                               (comparison_registry['mode_a'] == mode) & (comparison_registry['eval_data_a'] == eval_data) &
                               (((comparison_registry['arch_a'] == arch1) & (comparison_registry['arch_b'] == arch2)) |
                                ((comparison_registry['arch_a'] == arch2) & (comparison_registry['arch_b'] == arch1)))]
  if len(match) == 0:
    return None
  row = match.iloc[0]
  sign = 1 if row['arch_a'] == arch1 else -1
  result = stats_corrected[stats_corrected['comparison_id'] == row['comparison_id']].iloc[0]
  return {
      'sensitivity90_diff': sign * result['bootstrap_sensitivity90_diff'],  # positive = arch1 ahead
      'sensitivity90_significant': result['bootstrap_sensitivity90_p_significant'],
      'macro_f1_diff': sign * result['bootstrap_macro_f1_diff'],
      'macro_f1_significant': result['bootstrap_macro_f1_p_significant'],
  }

arch_pairs = [('resnet50', 'efficientnet_b0'), ('resnet50', 'swin_tiny'), ('efficientnet_b0', 'swin_tiny')]
pairwise_rows = []
for mode in ['TL', 'scratch']:
  for arch1, arch2 in arch_pairs:
    for eval_data in ['bus_uclm', 'breast']:
      comp = find_arch_comparison(arch1, arch2, mode, eval_data)
      pairwise_rows.append({
          'mode': mode, 'arch1': arch1, 'arch2': arch2, 'eval_data': eval_data,
          'sensitivity90_diff_(arch1-arch2)': comp['sensitivity90_diff'],
          'sensitivity90_significant': comp['sensitivity90_significant'],
      })

pairwise_df = pd.DataFrame(pairwise_rows)
print(pairwise_df.to_string(index=False))

### 7.3 Tier 2: Technical Tiebreak



In [ ]:
def get_busi_test_technical(architecture, mode):
  row = technical_df[(technical_df['architecture'] == architecture) & (technical_df['mode'] == mode) &
                      (technical_df['arm'] == 'inpainted') & (technical_df['eval_data'] == 'busi_test')].iloc[0]
  return row['macro_f1'], row['recall_malignant']

tier2_rows = []
for architecture in ['resnet50', 'efficientnet_b0', 'swin_tiny']:
  for mode in ['TL', 'scratch']:
    f1, sens = get_busi_test_technical(architecture, mode)
    tier2_rows.append({'architecture': architecture, 'mode': mode, 'busi_test_macro_f1': f1, 'busi_test_malignant_recall': sens})

tier2_df = pd.DataFrame(tier2_rows)
print(tier2_df.to_string(index=False))

best_tl = tier2_df[tier2_df['mode'] == 'TL'].sort_values(['busi_test_macro_f1', 'busi_test_malignant_recall'], ascending=False).iloc[0]
best_scratch = tier2_df[tier2_df['mode'] == 'scratch'].sort_values(['busi_test_macro_f1', 'busi_test_malignant_recall'], ascending=False).iloc[0]

print(f"\nTier 2 tiebreak pick, TL: {best_tl['architecture']} (macro_f1={best_tl['busi_test_macro_f1']:.3f})")
print(f"Tier 2 tiebreak pick, scratch: {best_scratch['architecture']} (macro_f1={best_scratch['busi_test_macro_f1']:.3f})")
print("\nNOTE: both picks are tiebreaks among architectures already shown statistically indistinguishable")
print("at Tier 1 -- not proven winners. Report accordingly, every time these are referenced.")

same_architecture = best_tl['architecture'] == best_scratch['architecture']
print(f"\nSame architecture for both modes: {same_architecture}")

### 7.4 Final Synthesis

Pulls the real, already tested `resnet50` TL-vs-scratch result. Assembles one summary object carrying every caveat from all three tiers.

In [ ]:
resnet50_tl_vs_scratch = tl_check[tl_check['arch_a'] == 'resnet50'][
    ['eval_data_a', 'bootstrap_macro_f1_diff', 'bootstrap_macro_f1_p_adj', 'bootstrap_macro_f1_p_significant']
].sort_values('eval_data_a')
print(resnet50_tl_vs_scratch.to_string(index=False))

resnet50_any_significant = resnet50_tl_vs_scratch['bootstrap_macro_f1_p_significant'].any()
resnet50_all_favor_tl = (resnet50_tl_vs_scratch['bootstrap_macro_f1_diff'] > 0).all()
print(f'\nresnet50 TL numerically ahead on all 3 datasets: {resnet50_all_favor_tl}')
print(f'resnet50 TL significantly ahead on any dataset: {resnet50_any_significant}')

In [ ]:
best_model_summary = {
    'tier0': {
        'candidates_evaluated': 6, 'candidates_passing': 0,
        'finding': 'All 6 candidates (3 architectures x 2 modes, inpainted arm) fail the pre-specified '
                   '15% malignant-FNR safety floor on both external datasets. Internal (busi_test) FNR is '
                   'acceptable for 5/6 models. No candidate is clinically safe by this criterion.',
    },
    'tier1': {
        'pairwise_comparisons_tested': 12, 'significant_comparisons': 0,
        'finding': 'No architecture significantly outperforms another in external generalization, in '
                   'either mode, on either external dataset. The TL descriptive ranking (efficientnet_b0 '
                   'nominally first) is an average of two individually non-significant, opposite-direction '
                   'differences (ahead on breast, behind on bus_uclm) -- not a real winner.',
    },
    'tier2': {
        'tl_pick': 'resnet50', 'scratch_pick': 'resnet50',
        'caveat': 'Tiebreak among architectures already shown statistically indistinguishable at Tier 1. '
                  'Selected on busi_test technical metrics only -- not evidence of superiority.',
    },
    'tl_vs_scratch_for_picked_architecture': {
        'architecture': 'resnet50',
        'tl_numerically_ahead_all_datasets': bool(resnet50_all_favor_tl),
        'tl_significantly_ahead_any_dataset': bool(resnet50_any_significant),
        'finding': 'resnet50/TL outperforms resnet50/scratch numerically on all 3 datasets (macro-F1 and '
                   'malignant recall), but the difference is not statistically significant on any of them '
                   'after correction -- unlike efficientnet_b0 and swin_tiny, where TL\'s advantage over '
                   'scratch IS significant. resnet50 is the exception to the RQ1 pattern, not confirmation of it.',
    },
    'overall_caveat': 'No model in this study meets the pre-specified clinical safety floor. Any "best '
                       'model" language below refers to relative standing among candidates that already '
                       'failed that floor, not a deployment recommendation.',
}

for tier, content in best_model_summary.items():
  print(f'\n--- {tier} ---')
  if isinstance(content, dict):
    for k, v in content.items():
      print(f'{k}: {v}')
  else:
    print(content)

In [ ]:
import json

required_keys = {'tier0', 'tier1', 'tier2', 'tl_vs_scratch_for_picked_architecture', 'overall_caveat'}
if set(best_model_summary.keys()) != required_keys:
  raise ValueError(f'best_model_summary missing expected keys: {required_keys - set(best_model_summary.keys())}')

if best_model_summary['tier0']['candidates_passing'] != 0:
  raise ValueError('tier0 candidates_passing does not match the confirmed 0/6 result -- do not save a stale summary.')

with open(EVAL_DIR / 'best_model_summary.json', 'w') as f:
  json.dump(best_model_summary, f, indent=2)

print(f'Saved to {EVAL_DIR / "best_model_summary.json"}')
print('\nSection 7 complete.')

### 7.5 Ablation Study: Clever Hans Hypothesis Test

Direct test, using 54 `arm_vs_arm` exploratory comparisons already computed and BH-corrected. Framed around the actual mechanism: if annotated/masked models relied on artefact shortcuts, that reliance cannot transfer to external datasets. So the direct test is whether inpainted significantly outperforms annotated/masked specifically on external data. Internal comparisons are included too, to check whether the original premise (smoother training-time loss curves for annotated/masked) has real statistical backing or was only ever a training-dynamics observation.

In [ ]:
def get_arm_comparison(architecture, mode, eval_data, reference_arm, other_arm):
  match = comparison_registry[(comparison_registry['comparison_type'] == 'arm_vs_arm') &
                               (comparison_registry['arch_a'] == architecture) & (comparison_registry['mode_a'] == mode) &
                               (comparison_registry['eval_data_a'] == eval_data) &
                               (((comparison_registry['arm_a'] == reference_arm) & (comparison_registry['arm_b'] == other_arm)) |
                                ((comparison_registry['arm_a'] == other_arm) & (comparison_registry['arm_b'] == reference_arm)))]
  row = match.iloc[0]
  sign = 1 if row['arm_a'] == reference_arm else -1  # always returns reference_arm - other_arm
  result = stats_corrected[stats_corrected['comparison_id'] == row['comparison_id']].iloc[0]
  return {
      'sensitivity90_diff': sign * result['bootstrap_sensitivity90_diff'],
      'sensitivity90_significant': result['bootstrap_sensitivity90_p_significant'],
      'macro_f1_diff': sign * result['bootstrap_macro_f1_diff'],
      'macro_f1_significant': result['bootstrap_macro_f1_p_significant'],
  }

In [ ]:
clever_hans_rows = []
for architecture in ['resnet50', 'efficientnet_b0', 'swin_tiny']:
  for mode in ['TL', 'scratch']:
    for eval_data in EVAL_DATASETS:
      for other_arm in ['annotated', 'masked']:
        comp = get_arm_comparison(architecture, mode, eval_data, 'inpainted', other_arm)
        clever_hans_rows.append({
            'architecture': architecture, 'mode': mode, 'eval_data': eval_data,
            'comparison': f'inpainted_vs_{other_arm}',
            'is_external': eval_data in ['bus_uclm', 'breast'],
            'sensitivity90_diff_(inpainted-other)': comp['sensitivity90_diff'],
            'sensitivity90_significant': comp['sensitivity90_significant'],
            'macro_f1_diff_(inpainted-other)': comp['macro_f1_diff'],
            'macro_f1_significant': comp['macro_f1_significant'],
        })

clever_hans_df = pd.DataFrame(clever_hans_rows)
print(f'{len(clever_hans_df)} rows (expected 36: 3 arch x 2 mode x 3 eval_data x 2 arm-pairs)')
print(clever_hans_df.to_string(index=False))

In [ ]:
internal = clever_hans_df[~clever_hans_df['is_external']]
external = clever_hans_df[clever_hans_df['is_external']]

print('INTERNAL (busi_test) -- tests the original premise (smoother training curves -> real internal advantage?):')
print(f"  Significant (either metric): {(internal['sensitivity90_significant'] | internal['macro_f1_significant']).sum()} / {len(internal)}")

print('\nEXTERNAL (bus_uclm, breast) -- direct Clever Hans test:')
sig_external = external[external['sensitivity90_significant'] | external['macro_f1_significant']]
print(f"  Significant (either metric): {len(sig_external)} / {len(external)}")
if len(sig_external) > 0:
  print('\n  Significant external rows:')
  print(sig_external[['architecture', 'mode', 'eval_data', 'comparison', 'sensitivity90_diff_(inpainted-other)', 'macro_f1_diff_(inpainted-other)']].to_string(index=False))
  confirming = (sig_external['sensitivity90_diff_(inpainted-other)'] > 0) | (sig_external['macro_f1_diff_(inpainted-other)'] > 0)
  print(f"\n  Of these, inpainted significantly AHEAD (Clever-Hans-confirming direction): {confirming.sum()}")
  print(f"  Of these, inpainted significantly BEHIND (disconfirming direction): {(~confirming).sum()}")
else:
  print('  None significant -- no external comparison in either direction survives correction.')

In [ ]:
def classify_direction(row):
  sig_diffs = []
  if row['sensitivity90_significant']:
    sig_diffs.append(row['sensitivity90_diff_(inpainted-other)'])
  if row['macro_f1_significant']:
    sig_diffs.append(row['macro_f1_diff_(inpainted-other)'])
  if all(d > 0 for d in sig_diffs):
    return 'confirming'
  elif all(d < 0 for d in sig_diffs):
    return 'disconfirming'
  else:
    return 'mixed'

sig_external = sig_external.copy()
sig_external['direction'] = sig_external.apply(classify_direction, axis=1)
print(sig_external['direction'].value_counts())
print(sig_external[['architecture', 'mode', 'eval_data', 'comparison', 'macro_f1_diff_(inpainted-other)', 'direction']].to_string(index=False))

In [ ]:
clever_hans_df.to_csv(EVAL_DIR / 'ablation_arm_comparisons.csv', index=False)

direction_counts = sig_external['direction'].value_counts().to_dict()
clever_hans_summary = {
    'total_comparisons': 36, 'internal_significant': 0,
    'external_significant': len(sig_external), 'external_total': 24,
    'direction_among_significant': direction_counts,
    'finding': 'No significant arm effect internally (0/12). Externally, 9/24 significant, split roughly '
               'evenly confirming vs. disconfirming (5 vs 4) once classified by which specific metric was '
               'significant. Eight of the significant results are on bus_uclm (n=629); the remaining one '
               'is on breast (n=256). '
               'sensitivity90 never reaches significance anywhere in this table; only macro_f1 ever does. Overall: '
               'inconclusive on the Clever Hans hypothesis -- neither confirmed nor disconfirmed.',
}

with open(EVAL_DIR / 'ablation_clever_hans_summary.json', 'w') as f:
  json.dump(clever_hans_summary, f, indent=2)

print(f'Saved: {EVAL_DIR / "ablation_arm_comparisons.csv"} (36 rows)')
print(f'Saved: {EVAL_DIR / "ablation_clever_hans_summary.json"}')

## 8. Inference Latency & Computational Cost

### 8.1 Load & Parse Latency Data

In [ ]:
import ast

def parse_latencies(val):
  return ast.literal_eval(val) if isinstance(val, str) else val

latency_df = latency_df.copy()
latency_df['batch_latencies_ms'] = latency_df['batch_latencies_ms'].apply(parse_latencies)
print(f"Parsed OK, type: {type(latency_df['batch_latencies_ms'].iloc[0])}")

### 8.2 Per-Model Inference Latency

Per-image latency computed per-batch (each batch's latency divided by its own true size, since the final batch of most datasets is smaller than 16), then aggregated via median, not mean: `cudnn.benchmark=True` triggers a one-time algorithm-search cost the first time each architecture encounters a new input shape in this process, producing a single extreme outlier batch per architecture (confirmed: scratch/annotated specifically, the first model of each architecture in `groupby`'s sorted iteration order). Median is robust to that single outlier without needing to identify or exclude it explicitly.

In [ ]:
def parse_latencies(val):
  return ast.literal_eval(val) if isinstance(val, str) else val

latency_df = latency_df.copy()
latency_df['batch_latencies_ms'] = latency_df['batch_latencies_ms'].apply(parse_latencies)

# n_images not saved to batch_latencies.csv; re-derive from technical_df,
# same source the original sweep used.
if 'n_images' not in latency_df.columns:
  latency_df = latency_df.merge(
      technical_df[['architecture', 'mode', 'arm', 'eval_data', 'n_images']],
      on=['architecture', 'mode', 'arm', 'eval_data'], how='left'
  )
  if latency_df['n_images'].isna().any():
    raise ValueError('n_images merge produced NaN -- check technical_df is loaded and complete.')

print(f"Parsed OK, n_images present: {'n_images' in latency_df.columns}")

def get_batch_sizes(n_images, batch_size=16):
  n_full = n_images // batch_size
  remainder = n_images % batch_size
  sizes = [batch_size] * n_full
  if remainder > 0:
    sizes.append(remainder)
  return sizes

def robust_ms_per_image(batch_latencies, n_images):
  batch_sizes = get_batch_sizes(n_images)
  if len(batch_sizes) != len(batch_latencies):
    raise ValueError(f'Batch count mismatch: {len(batch_latencies)} latencies vs {len(batch_sizes)} expected sizes for n_images={n_images}.')
  per_batch_per_image = [lat / sz for lat, sz in zip(batch_latencies, batch_sizes)]
  return float(np.median(per_batch_per_image))

latency_df['ms_per_image'] = latency_df.apply(lambda row: robust_ms_per_image(row['batch_latencies_ms'], row['n_images']), axis=1)

per_model_latency = latency_df.groupby(['architecture', 'mode', 'arm'])['ms_per_image'].agg(['median', 'std']).reset_index()
per_model_latency.columns = ['architecture', 'mode', 'arm', 'avg_ms_per_image', 'std_ms_per_image']
print(per_model_latency.to_string(index=False))

### 8.3 Verify & Save

Confirms every one of the 18 models has latency data before saving.

In [ ]:
cost_and_latency = results_summary.merge(per_model_latency, on=['architecture', 'mode', 'arm'], how='left')

if cost_and_latency['avg_ms_per_image'].isna().any():
  missing = cost_and_latency[cost_and_latency['avg_ms_per_image'].isna()][['architecture', 'mode', 'arm']]
  raise ValueError(f'Missing latency data:\n{missing.to_string(index=False)}')
if len(cost_and_latency) != 18:
  raise ValueError(f'Expected 18 rows, got {len(cost_and_latency)}')

cost_and_latency.to_csv(EVAL_DIR / 'computational_cost_full.csv', index=False)
print(cost_and_latency[['architecture', 'mode', 'arm', 'training_time_min', 'avg_ms_per_image']].to_string(index=False))
print(f'\nSaved: {EVAL_DIR / "computational_cost_full.csv"}')

## 9. Results Export & GitHub Artefacts

In [ ]:
print('Files in RESULTS/eval/:')
for f in sorted(EVAL_DIR.rglob('*')):
  if f.is_file():
    print(f'  {f.relative_to(EVAL_DIR)}')

print(f'\nException, lives in RESULTS/ directly: busi_scratch_normalization_stats.json')
print(f'  Exists: {(RESULTS / "busi_scratch_normalization_stats.json").exists()}')

### 9.1 README / Index

Generated from the actual files on disk (raw counts, column counts) so it can not silently drift from reality.

In [ ]:
FILE_REGISTRY = [
    {'path': 'predictions.csv', 'section': '3.4', 'description': 'Raw per-image predictions for all 54 (model, eval_data) combinations: true/predicted labels and full class-probability vectors. Source data for every metric and statistical test in this notebook.'},
    {'path': 'batch_latencies.csv', 'section': '3.4', 'description': 'Raw per-batch inference latency (ms) for all 54 combinations.'},
    {'path': 'technical_metrics.csv', 'section': '4.2', 'description': 'Technical metrics (4.7.1): accuracy, macro-F1/F2, per-class precision/recall/F1, AUC-ROC, PR-AUC, confusion matrix cell counts.'},
    {'path': 'confusion_matrices/', 'section': '4.3', 'description': 'One PNG per (model, eval_data) combination -- raw count + row-normalized percentage per cell. Filename: {architecture}_{mode}_{arm}_{eval_data}.png.'},
    {'path': 'clinical_metrics.csv', 'section': '5', 'description': 'Clinical metrics (4.7.2): malignant sensitivity at 85/90/95% specificity (with threshold), FNR, PPV/NPV with prevalence.'},
    {'path': 'statistical_results_raw.csv', 'section': '6.5-6.6', 'description': 'All 171 pairwise comparisons: McNemar, DeLong (per class), bootstrap CIs.'},
    {'path': 'ablation_arm_comparisons.csv', 'section': '7.5', 'description': 'All 36 arm-vs-arm comparisons underlying the Clever Hans hypothesis test.'},
    {'path': 'ablation_clever_hans_summary.json', 'section': '7.5', 'description': 'Clever Hans hypothesis verdict: inconclusive (5 confirming / 4 disconfirming among externally-significant comparisons; 0/12 significant internally).'},
    {'path': 'best_model_summary.json', 'section': '7.4', 'description': 'Tier 0-2 best-model determination. Key finding: 0/6 candidates pass the pre-specified clinical safety floor.'},
    {'path': 'computational_cost_full.csv', 'section': '8', 'description': 'Per-model training cost (from N03) and inference latency (median ms/image) side by side.'},
    {'path': 'statistical_results_corrected.csv', 'section': '6.6',
 'description': 'All 171 comparisons with Holm/BH-corrected p-values and significance flags added. '
                'Confirmatory family: Holm-Bonferroni. Exploratory family: Benjamini-Hochberg. '
                '14 separate correction groups (see Section 6.6 for full breakdown).'},
]


def describe_file(entry):
  rel = entry['path']
  if rel.endswith('/'):
    files = list((EVAL_DIR / rel.rstrip('/')).glob('*.png'))
    return f'{len(files)} files' if files else 'MISSING (0 files)'
  full_path = EVAL_DIR / rel
  if not full_path.exists():
    return 'MISSING'
  if rel.endswith('.csv'):
    df = pd.read_csv(full_path)
    return f'{len(df)} rows, {len(df.columns)} columns'
  content = json.load(open(full_path))
  return f'{len(content)} top-level keys'


readme_lines = ['# Notebook 04 -- Evaluation Results\n', 'Generated from actual file contents. See `04_evaluation.ipynb` for full methodology.\n']
for entry in FILE_REGISTRY:
  stat = describe_file(entry)
  readme_lines.append(f"## `{entry['path']}` (Section {entry['section']})\n{entry['description']}\n\n**Current state:** {stat}\n")

readme_lines.append('## Note: exception to this folder\n`busi_scratch_normalization_stats.json` lives in `RESULTS/` directly, not `RESULTS/eval/` -- see Section 9.1.\n')

readme_content = '\n'.join(readme_lines)
with open(EVAL_DIR / 'README.md', 'w') as f:
  f.write(readme_content)

print(readme_content)

### 9.2 Existence & Completeness Check

Final structural verification across every artefact the notebook produces.

In [ ]:
EXPECTED = {
    'predictions.csv': 17370, 'batch_latencies.csv': 54, 'technical_metrics.csv': 54,
    'clinical_metrics.csv': 54, 'statistical_results_raw.csv': 171,
    'ablation_arm_comparisons.csv': 36, 'computational_cost_full.csv': 18, 'statistical_results_corrected.csv': 171,
}

all_passed = True
for filename, expected_rows in EXPECTED.items():
  path = EVAL_DIR / filename
  if not path.exists():
    print(f'[FAIL] {filename}: MISSING')
    all_passed = False
    continue
  actual_rows = len(pd.read_csv(path))
  status = 'PASS' if actual_rows == expected_rows else 'FAIL'
  print(f'[{status}] {filename}: {actual_rows} rows (expected {expected_rows})')
  if status == 'FAIL':
    all_passed = False

n_cm = len(list((EVAL_DIR / 'confusion_matrices').glob('*.png')))
print(f'[{"PASS" if n_cm == 54 else "FAIL"}] confusion_matrices/: {n_cm} files (expected 54)')
all_passed &= (n_cm == 54)

for filename in ['ablation_clever_hans_summary.json', 'best_model_summary.json']:
  path = EVAL_DIR / filename
  ok = path.exists() and path.stat().st_size > 0
  print(f'[{"PASS" if ok else "FAIL"}] {filename}: {"present, non-empty" if ok else "MISSING or empty"}')
  all_passed &= ok

exception_path = RESULTS / 'busi_scratch_normalization_stats.json'
ok = exception_path.exists()
print(f'[{"PASS" if ok else "FAIL"}] busi_scratch_normalization_stats.json (RESULTS/): {"present" if ok else "MISSING"}')
all_passed &= ok

if not all_passed:
  raise ValueError('Section 9 completeness check failed -- see FAILs above.')
print('\nAll artifacts present and correctly sized.')

## 10. Post-Hoc: BUS-UCLM vs BrEaST Performance Asymmetry

Motivated by a pattern observed during Notebook 05's GradCAM analysis: 15 images where all 6 models correctly classified malignant on BrEaST vs. only 2 on BUS-UCLM. This cell checks whether the internal-to-external performance drop differs significantly and systematically between the two external datasets, using `internal_vs_external` comparisons already computed and corrected in Section 6. Post-hoc and exploratory; labeled as such throughout.

In [ ]:
# Pull the internal_vs_external comparisons for the inpainted arm only,
# separating busi_test->bus_uclm from busi_test->breast.
inpainted_ext = stats_corrected[
    (stats_corrected['comparison_type'] == 'internal_vs_external') &
    (stats_corrected['arm_a'] == 'inpainted')
].copy()

bus_uclm_comps = inpainted_ext[inpainted_ext['eval_data_b'] == 'bus_uclm']
breast_comps = inpainted_ext[inpainted_ext['eval_data_b'] == 'breast']

print(f'BUS-UCLM comparisons: {len(bus_uclm_comps)} (expected 6)')
print(f'BrEaST comparisons: {len(breast_comps)} (expected 6)')

display_cols = ['arch_a', 'mode_a', 'family',
                'bootstrap_macro_f1_diff', 'bootstrap_macro_f1_ci_lower',
                'bootstrap_macro_f1_ci_upper', 'bootstrap_macro_f1_p_adj',
                'bootstrap_macro_f1_p_significant',
                'bootstrap_sensitivity90_diff', 'bootstrap_sensitivity90_p_adj',
                'bootstrap_sensitivity90_p_significant']

print('\n=== Internal vs BUS-UCLM (inpainted arm) ===')
print('Note: diff = internal - external, so positive = internal better')
print(bus_uclm_comps[display_cols].sort_values(['arch_a', 'mode_a']).to_string(index=False))

print('\n=== Internal vs BrEaST (inpainted arm) ===')
print(breast_comps[display_cols].sort_values(['arch_a', 'mode_a']).to_string(index=False))

In [ ]:
# Summary: how many of each reach significance, and what's the
# magnitude difference between the two external datasets' drops?
print('=== Summary ===')
for label, df in [('BUS-UCLM', bus_uclm_comps), ('BrEaST', breast_comps)]:
    sig_f1 = df['bootstrap_macro_f1_p_significant'].sum()
    sig_sens = df['bootstrap_sensitivity90_p_significant'].sum()
    mean_f1_drop = df['bootstrap_macro_f1_diff'].mean()
    mean_sens_drop = df['bootstrap_sensitivity90_diff'].mean()
    print(f'\n{label}:')
    print(f'  Significant (macro-F1): {sig_f1}/6')
    print(f'  Significant (sensitivity90): {sig_sens}/6')
    print(f'  Mean macro-F1 drop (internal - external): {mean_f1_drop:.3f}')
    print(f'  Mean sensitivity90 drop (internal - external): {mean_sens_drop:.3f}')

# Direct comparison of drop magnitudes between the two external datasets
print('\n=== Per-model: BUS-UCLM drop vs BrEaST drop ===')
merged_drops = bus_uclm_comps[['arch_a','mode_a','bootstrap_macro_f1_diff']].merge(
    breast_comps[['arch_a','mode_a','bootstrap_macro_f1_diff']],
    on=['arch_a','mode_a'], suffixes=('_bus_uclm','_breast')
).copy()
merged_drops['bus_uclm_harder_by'] = merged_drops['bootstrap_macro_f1_diff_bus_uclm'] - merged_drops['bootstrap_macro_f1_diff_breast']
merged_drops['arch_a'] = merged_drops['arch_a'].map({'resnet50':'ResNet50','efficientnet_b0':'EfficientNet-B0','swin_tiny':'Swin-T'})
print(merged_drops[['arch_a','mode_a','bootstrap_macro_f1_diff_bus_uclm',
                      'bootstrap_macro_f1_diff_breast','bus_uclm_harder_by']].to_string(index=False))
print(f'\nMean additional drop for BUS-UCLM vs BrEaST: {merged_drops["bus_uclm_harder_by"].mean():.3f}')
print(f'All models show larger drop on BUS-UCLM: {(merged_drops["bus_uclm_harder_by"] > 0).all()}')